# 🔗 34 · Multimodal single-cell integration: RNA, protein and chromatin
### CITE-seq and multiome data, modality-specific normalisation (CLR, background-aware ADT scaling, TF-IDF + LSI), early, late and intermediate integration (CCA, weighted nearest neighbours), cross-modal prediction, enhancer–gene links and joint factor models — built from scratch and graded against states that only one modality can see

**BiologyToML** · Level 2 · Research frontier · Part G — Cells and tissues · Module 34 of 44

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/34_multimodal_integration.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, pandas, Matplotlib, scikit-learn (k-NN, ridge, Gaussian mixtures, randomized SVD, ARI/AUROC), NetworkX (Louvain) |
| **Prerequisites** | Level 1 (especially Modules 07, 12–14 and 18) and [Module 30](30_single_cell_rnaseq.ipynb) (the scRNA-seq pipeline); [Module 25](25_variational_inference_vae.ipynb) helps for Section 7 |
| **Estimated time** | 170 min (reading + running + the ✏️ exercises) |
| **Builds on** | [07 · Ridge and lasso](07_ridge_lasso_sparse_regression.ipynb), [08 · k-NN](08_knn_naive_bayes.ipynb), [13 · Mixture models](13_mixture_models_em.ipynb), [14 · PCA](14_pca_dimensionality_reduction.ipynb), [18 · Gaussian graphical models](18_gaussian_graphical_models.ipynb), [23 · Model selection](23_model_selection.ipynb), [25 · VAEs](25_variational_inference_vae.ipynb), [30 · scRNA-seq](30_single_cell_rnaseq.ipynb), [31 · Integration](31_batch_integration.ipynb) |
| **Research frontier covered** | CITE-seq and single-cell multiome, modality-specific normalisation, weighted-nearest-neighbour and CCA integration, cross-modal prediction, enhancer–gene linking — see [`FRONTIER_MAP.md`](../FRONTIER_MAP.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> **All data are SIMULATED** with the toolkit's `simulate_multiome`: 2,500 PBMC-like cells, each measured three ways —
> RNA (800 genes), 15 surface proteins (CITE-seq antibody-derived tags, including an isotype control) and chromatin
> accessibility at 2,000 peaks (binary, scATAC-like). The hidden truth contains each cell's type and three states that
> are invisible in RNA by construction: naive vs memory T cells and regulatory T cells (visible in protein) and "primed"
> monocytes with open chromatin at interferon genes that are not yet transcribed (visible in chromatin), plus the gene
> each peak regulates. Real data add many more features, much sparser chromatin, ambient antibodies and no answer key —
> see Section 8.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** what CITE-seq antibody tags, scATAC-seq and joint multiome assays measure, and which artefacts each modality carries (antibody background, binary sparsity, depth).
2. **Normalise** each modality appropriately: centred log-ratio and a background-aware (DSB-style) scaling for proteins, TF-IDF + latent semantic indexing for chromatin — and **show** why the first LSI component is dropped.
3. **Measure** what each modality sees on its own, with the adjusted Rand index and a neighbourhood AUROC for states hidden from RNA.
4. **Implement** early, late and intermediate integration — canonical correlation analysis derived from the SVD and a simplified weighted-nearest-neighbour (WNN) graph — and **grade** them against the full truth.
5. **Predict** proteins from RNA with ridge and k-NN regression, and **explain** which proteins cannot be predicted and why that justifies measuring them.
6. **Link** enhancers to genes by peak–gene correlation across metacells, **quantify** confounding by cell type, and **show** how a genomic-distance prior changes precision.
7. **Decompose** multimodal variation into shared and modality-specific factors (the idea behind MOFA+ and multimodal VAEs).
8. **Connect** every step to Seurat WNN, Signac, ArchR, MOFA+ and totalVI, and to public multimodal data sets.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit | 5 min |
| 1 · Three modalities in one cell | assays, the simulated experiment, what the raw data look like | 15 min |
| 2 · Normalising each modality | RNA recap; CLR and DSB-lite for proteins; TF-IDF + LSI for chromatin; 🧪 artefacts | 30 min |
| 3 · One modality at a time | clustering and neighbourhoods per modality; 🧪 what each modality sees | 15 min |
| 4 · Integration strategies | early, late, CCA, WNN-lite; 🧪 ARI against the full truth | 35 min |
| 5 · Cross-modal prediction | proteins from RNA; gene activity from chromatin; 🧪 R² per protein | 20 min |
| 6 · Enhancer–gene links | metacells, peak–gene correlation, confounding, distance priors; 🧪 AUROC/precision | 20 min |
| 7 · Joint latent-variable models | MOFA+ and multimodal VAEs; a weighted joint PCA; 🧪 shared vs private factors | 15 min |
| 8 · 🧭 From toy to real data | tools, data sets, what goes wrong | 10 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 5 min |

## 🧫 The biological question

Two T cells can have nearly identical transcriptomes and still be very different cells. A **naive** T cell has never met
its antigen; a **memory** T cell has, and responds faster and more strongly the second time. Immunologists have told
them apart for decades with antibodies against two forms of the surface protein CD45: **CD45RA** on naive cells,
**CD45RO** on memory cells. Both are produced from the *same gene*, *PTPRC*, by alternative splicing — so a
gene-level RNA count cannot distinguish them, while an antibody can. **Regulatory T cells** (Tregs), which keep immune
responses in check, are recognised by high surface **CD25**; its transcript is often barely detected in droplet
scRNA-seq.

Chromatin tells a third story. Before a gene is transcribed, the DNA around its promoter and enhancers must be opened
up. Cells can be **primed** — their regulatory elements open, the genes still silent — so that a later signal produces a
faster response. Such a state is invisible in RNA and in surface proteins and visible only in **chromatin
accessibility**. Ma *et al.* (2020) measured RNA and chromatin in the same cells (SHARE-seq) and found that accessibility
at key regulatory regions often changes *before* the expression of the nearby genes ("chromatin potential").

Since 2017, assays have measured several modalities in the **same cell**: CITE-seq (Stoeckius *et al.* 2017) and REAP-seq
(Peterson *et al.* 2017) add surface proteins to scRNA-seq; joint RNA + ATAC methods (SHARE-seq, 10x Genomics Multiome)
add chromatin; trimodal methods measure all three. Hao *et al.* (2021) profiled more than 200,000 human blood cells with
CITE-seq and over 200 antibodies and introduced **weighted-nearest-neighbour** analysis, which learns for every cell how
much to trust each modality.

Our SIMULATED data mimic a trimodal PBMC experiment. The questions of this notebook are the questions of every
multimodal study:

1. How should each modality be normalised, given its very different noise?
2. What does each modality see on its own — and how do we combine them so that the joint analysis sees *everything*?
3. What can one modality predict about another — and what can it not?
4. Which regulatory elements control which genes?

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("sklearn", "scikit-learn"), ("networkx", "networkx")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import time

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import adjusted_rand_score, average_precision_score, precision_recall_curve, roc_auc_score
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import KNeighborsRegressor, NearestNeighbors
from sklearn.utils.extmath import randomized_svd

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
`simulate_multiome` generates the trimodal experiment with its hidden truth (it calls `simulate_scrna` for the RNA part, which is copied too), `ADT_PANEL` is the antibody panel (which protein, which encoding gene, which cell types express it), `log_normalize` and `pca` are the Module 30 workhorses, and `plot_embedding` scatters 2-D embeddings by label. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MITO_GENES = ["MT-ND1", "MT-ND2", "MT-CO1", "MT-CO2", "MT-ATP8", "MT-ATP6", "MT-CO3", "MT-ND3",
              "MT-ND4L", "MT-ND4", "MT-ND5", "MT-ND6", "MT-CYB"]


INTERFERON_GENES = ["ISG15", "IFI6", "IFIT1", "IFIT3", "MX1", "OAS1", "IFI44L", "RSAD2", "IRF7", "STAT1"]


CYCLING_GENES = ["MKI67", "TOP2A", "CCNB1", "CDK1", "UBE2C", "BIRC5", "TYMS", "PCNA"]


PBMC_TYPES = {
    "CD4 T": ("lymphoid", ["CD3E", "CD3D", "IL7R", "CD4"], 0.5),
    "CD8 T": ("lymphoid", ["CD3E", "CD3D", "CD8A", "CD8B"], 0.5),
    "NK": ("lymphoid", ["NKG7", "GNLY", "KLRD1", "PRF1"], 2.2),
    "B": ("lymphoid", ["MS4A1", "CD79A", "CD79B", "CD19"], 2.4),
    "Monocyte": ("myeloid", ["CD14", "LYZ", "S100A8", "S100A9"], 2.6),
    "Dendritic": ("myeloid", ["FCER1A", "CST3", "CLEC10A", "HLA-DQA1"], 2.2),
}


def sample_negative_binomial(rng: np.random.Generator, mu, theta):
    """Negative-binomial counts with mean ``mu`` and inverse dispersion ``theta``.

    Variance = mu + mu²/theta (large theta → Poisson).  Drawn as a Gamma–Poisson mixture:
    each count is Poisson with a rate that itself varies from cell to cell.
    """
    mu = np.asarray(mu, float)
    theta = np.broadcast_to(np.asarray(theta, float), mu.shape)
    return rng.poisson(rng.gamma(theta, np.where(mu > 0, mu, 0.0) / theta)).astype(np.int32)


def make_expression_programs(rng: np.random.Generator, n_genes: int, cell_types: dict,
                             n_extra_markers: int = 20, extra_names=()):
    """Log-scale expression profiles (one row per cell type) for a simulated tissue.

    ``cell_types`` maps a type name to (lineage, named marker genes, marker strength).  Every
    type gets its named markers plus ``n_extra_markers`` anonymous ones; types of the same
    lineage share 20 lineage genes.  ``extra_names`` are reserved gene names (placed after the
    markers) that the caller gives its own behaviour.  Returns ``(log_profiles, genes, marker_of)``.
    """
    named = []
    for _, markers, _ in cell_types.values():
        named += [g for g in markers if g not in named]
    reserved = named + [g for g in extra_names if g not in named]
    if len(reserved) > n_genes:
        raise ValueError("n_genes is too small for the requested marker genes")
    genes = np.array(reserved + [f"Gene{j:04d}" for j in range(1, n_genes - len(reserved) + 1)], dtype=object)
    free = np.arange(len(reserved), n_genes)                       # anonymous genes available
    base = rng.normal(0.0, 2.0, n_genes)                           # gene-specific baseline (log scale)
    base[:len(named)] -= 2.0                                       # named markers are low elsewhere
    profiles = np.tile(base, (len(cell_types), 1)) + rng.normal(0, 0.15, (len(cell_types), n_genes))
    marker_of = np.array([""] * n_genes, dtype=object)
    lineages = sorted({v[0] for v in cell_types.values()})
    lineage_genes = {lin: rng.choice(free, 20, replace=False) for lin in lineages}
    for t, (name, (lineage, markers, strength)) in enumerate(cell_types.items()):
        profiles[t, lineage_genes[lineage]] += rng.normal(1.2, 0.3, 20)
        idx = [int(np.flatnonzero(genes == g)[0]) for g in markers]
        profiles[t, idx] += strength + 2.0 + rng.normal(0, 0.2, len(idx))
        extra = rng.choice(free, n_extra_markers, replace=False)
        profiles[t, extra] += rng.normal(0.6 + strength / 2, 0.3, n_extra_markers)
        for j in list(idx) + list(extra):
            marker_of[j] = name if marker_of[j] == "" else marker_of[j] + "/" + name
    return profiles, genes, marker_of


def simulate_scrna(rng: np.random.Generator, n_cells: int = 3000, n_genes: int = 1000, cell_types=None,
                   proportions=None, n_donors: int = 6, n_batches: int = 2, confounded: bool = False,
                   depth: float = 1500, batch_effect: float = 0.25, donor_effect: float = 0.15,
                   stim_effect: float = 1.0, theta: float = 8.0, doublet_rate: float = 0.0,
                   damaged_rate: float = 0.0, cycling_rate: float = 0.0, drop_type_in_batch=None) -> dict:
    """Simulate single-cell RNA-seq UMI counts for a PBMC-like sample collection.

    Design: ``n_donors`` donors, the first half *control* and the second half *stimulated*
    (an interferon response in every cell type plus a monocyte-specific response).  Donor
    ``d`` is sequenced in batch ``d % n_batches`` — or, with ``confounded=True``, every control
    donor in batch 1 and every stimulated donor in batch 2 (a design flaw to learn from).
    Counts are negative binomial around ``depth`` UMIs per cell.  Optional artefacts for
    quality control: doublets, damaged cells (high mitochondrial fraction), cycling cells.
    ``drop_type_in_batch=("B", 1)`` removes B cells from batch 1 (index 1, i.e. "batch2").

    Expression probabilities are renormalised within each cell (a fixed-size library, as in
    sequencing), so in stimulated cells the genes with ``true_logfc == 0`` lose a few per cent of
    their share (about −0.04 on the natural-log scale) to the induced genes: a compositional
    effect that is also present in real data.

    Returns a dict with ``counts`` (cells × genes, int32), ``obs`` (per-cell table with the
    true cell type, donor, batch, condition, …), ``var`` (per-gene table), ``true_logfc``
    (cell type × gene stimulation effect, natural log) and ``log_profiles``.
    """
    cell_types = PBMC_TYPES if cell_types is None else cell_types
    names = list(cell_types)
    T = len(names)
    p = np.full(T, 1 / T) if proportions is None else np.asarray(proportions, float) / np.sum(proportions)
    extra = MITO_GENES + INTERFERON_GENES + CYCLING_GENES
    profiles, genes, marker_of = make_expression_programs(rng, n_genes, cell_types, extra_names=extra)
    G = len(genes)
    is_mito, is_ifn, is_cyc = (np.isin(genes, MITO_GENES), np.isin(genes, INTERFERON_GENES),
                               np.isin(genes, CYCLING_GENES))
    profiles[:, is_mito] = rng.normal(3.0, 0.3, is_mito.sum())          # ≈ 4 % of UMIs are mitochondrial
    profiles[:, is_ifn | is_cyc] = rng.normal(-1.0, 0.4, (T, (is_ifn | is_cyc).sum()))
    # stimulation: interferon genes up everywhere + 40 monocyte-specific responders (up or down)
    true_logfc = np.zeros((T, G))
    true_logfc[:, is_ifn] = stim_effect * rng.normal(1.6, 0.3, is_ifn.sum())
    if "Monocyte" in names:
        responders = rng.choice(np.flatnonzero((marker_of == "") & ~is_mito & ~is_ifn & ~is_cyc), 40, replace=False)
        true_logfc[names.index("Monocyte"), responders] = stim_effect * rng.choice([-1, 1], 40) * rng.normal(1.2, 0.3, 40)
    # donors, batches, conditions
    donor = rng.integers(0, n_donors, n_cells)
    stimulated = donor >= n_donors // 2
    batch = stimulated.astype(int) if confounded else donor % n_batches
    batch = np.minimum(batch, n_batches - 1)
    cell_type = rng.choice(T, n_cells, p=p)
    if drop_type_in_batch is not None:
        t_drop, b_drop = names.index(drop_type_in_batch[0]), drop_type_in_batch[1]
        bad = (cell_type == t_drop) & (batch == b_drop)
        cell_type[bad] = rng.choice(np.delete(np.arange(T), t_drop), bad.sum(), p=np.delete(p, t_drop) / (1 - p[t_drop]))
    batch_shift = rng.normal(0, batch_effect, (n_batches, G))
    batch_depth = np.exp(rng.normal(0, 0.3, n_batches))
    donor_shift = rng.normal(0, donor_effect, (n_donors, G))
    log_rate = (profiles[cell_type] + batch_shift[batch] + donor_shift[donor]
                + true_logfc[cell_type] * stimulated[:, None])
    cycling = rng.random(n_cells) < cycling_rate
    log_rate[np.ix_(cycling, np.flatnonzero(is_cyc))] += 3.0
    damaged = rng.random(n_cells) < damaged_rate
    log_rate[np.ix_(damaged, np.flatnonzero(is_mito))] += 2.2
    lib = depth * batch_depth[batch] * np.exp(rng.normal(0, 0.35, n_cells))
    lib[damaged] *= 0.35
    prob = np.exp(log_rate - log_rate.max(axis=1, keepdims=True))
    prob /= prob.sum(axis=1, keepdims=True)
    theta_g = theta * np.exp(rng.normal(0, 0.5, G))
    mu = lib[:, None] * prob
    doublet = rng.random(n_cells) < doublet_rate
    partner = rng.integers(0, n_cells, n_cells)
    mu[doublet] = 0.9 * mu[doublet] + 0.9 * mu[partner[doublet]]       # two cells in one droplet
    counts = sample_negative_binomial(rng, mu, theta_g)
    obs = pd.DataFrame({
        "cell_type": np.where(doublet, "Doublet", np.array(names, dtype=object)[cell_type]),
        "true_type": np.array(names, dtype=object)[cell_type],
        "lineage": np.array([cell_types[n][0] for n in names], dtype=object)[cell_type],
        "donor": [f"donor{d + 1}" for d in donor],
        "condition": np.where(stimulated, "stimulated", "control"),
        "batch": [f"batch{b + 1}" for b in batch],
        "n_counts": counts.sum(axis=1),
        "n_genes": (counts > 0).sum(axis=1),
        "pct_mito": 100 * counts[:, is_mito].sum(axis=1) / np.maximum(counts.sum(axis=1), 1),
        "is_doublet": doublet, "is_damaged": damaged, "is_cycling": cycling,
    }, index=[f"cell{i:05d}" for i in range(n_cells)])
    var = pd.DataFrame({"marker_of": marker_of, "is_mito": is_mito, "is_interferon": is_ifn,
                        "is_cycling": is_cyc, "theta": theta_g}, index=genes)
    return dict(counts=counts, obs=obs, var=var, log_profiles=pd.DataFrame(profiles, index=names, columns=genes),
                true_logfc=pd.DataFrame(true_logfc, index=names, columns=genes))


ADT_PANEL = {
    "CD3": ("CD3E", {"CD4 T": 1.0, "CD8 T": 1.0}),
    "CD4": ("CD4", {"CD4 T": 1.0, "Monocyte": 0.25, "Dendritic": 0.2}),
    "CD8": ("CD8A", {"CD8 T": 1.0}),
    "CD56": ("NCAM1", {"NK": 1.0}),
    "CD16": ("FCGR3A", {"NK": 0.9, "Monocyte": 0.3}),
    "CD19": ("CD19", {"B": 1.0}),
    "CD20": ("MS4A1", {"B": 1.0}),
    "CD14": ("CD14", {"Monocyte": 1.0}),
    "CD11c": ("ITGAX", {"Monocyte": 0.5, "Dendritic": 1.0}),
    "HLA-DR": ("HLA-DRA", {"B": 0.8, "Monocyte": 0.8, "Dendritic": 1.0}),
    "CD45RA": ("PTPRC", {"B": 0.9, "NK": 0.8}),          # also on naive T cells (set per cell below)
    "CD45RO": ("PTPRC", {}),                              # on memory T cells (set per cell below)
    "CD127": ("IL7R", {"CD4 T": 0.7}),
    "CD25": ("IL2RA", {}),                                # regulatory T cells only (set per cell below)
    "IgG1-isotype": ("", {}),                             # negative control: background only
}


def simulate_multiome(rng: np.random.Generator, n_cells: int = 2500, n_genes: int = 800,
                      n_peaks: int = 2000) -> dict:
    """Simulate a trimodal single-cell experiment: RNA + surface protein (CITE-seq) + chromatin (ATAC).

    Some biology is visible in only one modality, which is why multimodal assays exist:
    naive vs memory T cells and regulatory T cells differ mainly in *protein* (CD45RA/CD45RO,
    CD25); a "primed" monocyte state has open chromatin at interferon genes that are not yet
    transcribed, so it is visible only in *ATAC*.  RNA comes from ``simulate_scrna`` (one
    batch, control donors).  Protein counts include ambient background (see the isotype
    control); ATAC is a binary cell × peak matrix with sparse, depth-dependent capture.

    Returns a dict with ``rna``, ``adt`` and ``atac`` matrices, ``obs`` (true cell type plus
    the hidden states), ``var`` (genes), ``proteins`` and ``peaks`` (each peak's linked gene).
    """
    sc = simulate_scrna(rng, n_cells=n_cells, n_genes=n_genes, n_donors=2, n_batches=1, stim_effect=0.0)
    obs, profiles, genes = sc["obs"].copy(), sc["log_profiles"], np.array(sc["var"].index)
    names = list(profiles.index)
    ttype = obs["true_type"].to_numpy()
    is_t = np.isin(ttype, ["CD4 T", "CD8 T"])
    memory = is_t & (rng.random(n_cells) < 0.5)
    treg = (ttype == "CD4 T") & (rng.random(n_cells) < 0.12)
    primed = (ttype == "Monocyte") & (rng.random(n_cells) < 0.35)
    obs["t_state"] = np.where(is_t, np.where(memory, "memory", "naive"), "")
    obs["is_treg"], obs["is_primed"] = treg, primed
    # surface proteins
    level = np.zeros((n_cells, len(ADT_PANEL)))
    for j, (prot, (_, by_type)) in enumerate(ADT_PANEL.items()):
        for t, v in by_type.items():
            level[ttype == t, j] = v
    pidx = list(ADT_PANEL)
    level[is_t & ~memory, pidx.index("CD45RA")] = 1.0
    level[memory, pidx.index("CD45RO")] = 1.0
    level[treg, pidx.index("CD25")] = 1.0
    size = np.exp(rng.normal(0, 0.3, n_cells))[:, None]
    adt = sample_negative_binomial(rng, size * (15 + 400 * level * rng.uniform(0.6, 1.4, len(pidx))), 6.0)
    # chromatin accessibility: peaks near genes open where the gene is (or is about to be) active
    active = (profiles.to_numpy() > profiles.to_numpy().mean(axis=0) + 0.8).astype(float)
    linked = rng.integers(0, len(genes), n_peaks)
    kind = rng.choice(["promoter", "enhancer", "background"], n_peaks, p=[0.35, 0.35, 0.30])
    linked_gene = np.where(kind == "background", "", genes[linked])
    # every interferon gene gets ten enhancer peaks (the primed-state signal)
    extra = np.repeat([np.flatnonzero(genes == g)[0] for g in INTERFERON_GENES], 10)
    slots = rng.choice(np.flatnonzero(kind == "background"), len(extra), replace=False)
    kind[slots], linked[slots], linked_gene[slots] = "enhancer", extra, genes[extra]
    ifn_peak = np.isin(linked_gene, INTERFERON_GENES)
    t_idx = np.array([names.index(t) for t in ttype])
    logit = rng.normal(-2.5, 0.8, n_peaks) + np.where(kind == "background", 0.0, 3.0 * active[t_idx][:, linked])
    logit[:, ifn_peak] += 3.5 * primed[:, None]
    capture = rng.beta(4, 6, n_cells)[:, None]
    atac = (rng.random((n_cells, n_peaks)) < capture / (1 + np.exp(-logit))).astype(np.int8)
    proteins = pd.DataFrame({"gene": [v[0] for v in ADT_PANEL.values()]}, index=pidx)
    peaks = pd.DataFrame({"kind": kind, "linked_gene": linked_gene, "is_interferon": ifn_peak},
                         index=[f"peak{p:05d}" for p in range(n_peaks)])
    return dict(rna=sc["counts"], adt=adt, atac=atac, obs=obs, var=sc["var"], proteins=proteins, peaks=peaks)


def log_normalize(counts, target_sum: float = 1e4):
    """Library-size normalisation + log: log(1 + target_sum · count / total count of the cell)."""
    counts = np.asarray(counts, float)
    total = counts.sum(axis=1, keepdims=True)
    return np.log1p(target_sum * counts / np.where(total > 0, total, 1.0))


def pca(X, n_components: int = 2):
    """Principal component analysis by SVD of the column-centred data.

    Returns ``(scores, components, explained_variance_ratio)``: scores are N × k, components k × J.
    """
    X = np.asarray(X, float)
    Xc = X - X.mean(axis=0)
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    var = S ** 2 / (len(X) - 1)
    return U[:, :n_components] * S[:n_components], Vt[:n_components], var[:n_components] / var.sum()


CLASS_COLORS = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#ff7f0e", "#8c564b"]


def plot_embedding(Z, labels, ax=None, title: str = "", s: float = 4, legend: bool = True, cmap=None):
    """Scatter a 2-D embedding (PCA, t-SNE, latent space, …) coloured by a categorical or numeric label."""
    Z = np.asarray(Z, float)
    labels = np.asarray(labels)
    ax = ax or plt.gca()
    if labels.dtype.kind in "fc":
        sc = ax.scatter(Z[:, 0], Z[:, 1], c=labels, s=s, cmap=cmap or "viridis", linewidths=0)
        plt.colorbar(sc, ax=ax, shrink=0.8)
    else:
        palette = plt.get_cmap("tab10" if len(np.unique(labels)) <= 10 else "tab20")
        for k, lab in enumerate(np.unique(labels)):
            m = labels == lab
            ax.scatter(Z[m, 0], Z[m, 1], s=s, color=palette(k % palette.N), label=str(lab), linewidths=0)
        if legend:
            ax.legend(markerscale=3, fontsize=7, frameon=False, loc="best")
    ax.set_xticks([]); ax.set_yticks([])
    if title:
        ax.set_title(title)
    return ax


print("🧰 Toolkit loaded:", ", ".join(['sample_negative_binomial', 'make_expression_programs', 'simulate_scrna', 'simulate_multiome', 'log_normalize', 'pca', 'plot_embedding']))

In [ ]:
# ── 0.1 Start the clock ──
T_START = time.time()

---
## 1 · Three modalities in one cell

> 📍 **Where we are.** [Module 30](30_single_cell_rnaseq.ipynb) turned one modality — RNA — into cell types. Here every
> cell carries three measurements of different physical things, with different noise. We first need to understand how
> each is made.

### 🔬 Assay box — CITE-seq, scATAC-seq and multiome

- **CITE-seq / REAP-seq (RNA + surface protein).** Before the cells enter the droplets, they are stained with a panel of
  antibodies (tens to a few hundred), each carrying a short DNA barcode with a poly-A tail. Inside the droplet the
  antibody barcodes are captured like mRNAs, so each cell gets **antibody-derived tag (ADT)** counts next to its RNA
  counts. ADT counts are *high* (hundreds per protein, far more than most mRNAs) but carry **background**: free-floating
  antibodies in the droplet (ambient) and antibodies that stick non-specifically. **Isotype controls** — antibodies of
  the same class but without a human target — measure that background.
- **scATAC-seq (chromatin accessibility).** The Tn5 transposase cuts DNA and inserts sequencing adapters, but only
  where the DNA is not wrapped around nucleosomes or bound by proteins — in "open" chromatin at active promoters and
  enhancers (ATAC = Assay for Transposase-Accessible Chromatin; Buenrostro *et al.* 2013, 2015 for single cells;
  Cusanovich *et al.* 2015). Fragments are aggregated into **peaks** (regions of a few hundred base pairs). Because a
  diploid cell has only two copies of each region, a peak is essentially **open or closed**: the cell × peak matrix is
  **binary** and extremely sparse (in real data typically only a few per cent of entries are non-zero).
- **Multiome (RNA + ATAC in the same nucleus)**, e.g. SHARE-seq (Ma *et al.* 2020) or the 10x Genomics Multiome kit, and
  trimodal methods (TEA-seq, DOGMA-seq, both 2021) that add antibodies. Joint measurement removes the need to *guess*
  which cell in one experiment corresponds to which in another — the integration of this notebook is integration of
  modalities within each cell, not of separate experiments ([Module 31](31_batch_integration.ipynb)).
- **Typical artefacts.** ADT: ambient and non-specific background, antibody-specific efficiency, cell size. ATAC: huge
  variation in the number of fragments per cell (depth), binary saturation, background "noise" peaks. RNA: everything
  from Module 30. Every modality needs its own normalisation (Section 2).

The schematic below is drawn with Matplotlib.

In [ ]:
# ── 1.1 What each modality reads out (schematic) ──
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.set_xlim(0, 11); ax.set_ylim(0, 3.2); ax.set_aspect("equal"); ax.axis("off")
ax.add_patch(plt.Circle((2.0, 1.6), 1.45, fc="#f7fbff", ec="#3182bd", lw=1.5))            # the cell
ax.add_patch(plt.Circle((2.0, 1.6), 0.65, fc="#fee6ce", ec="#e6550d", lw=1.2))            # the nucleus
xs = np.linspace(1.5, 2.5, 60)
ax.plot(xs, 1.6 + 0.12 * np.sin(18 * xs), color="0.3", lw=1)                             # DNA
for x0 in (1.62, 2.18):                                                                    # nucleosomes (closed)
    ax.add_patch(plt.Circle((x0, 1.6), 0.09, fc="0.55", ec="none"))
ax.annotate("", xy=(1.9, 1.75), xytext=(1.9, 2.05), arrowprops=dict(arrowstyle="-|>", color="#e6550d"))
ax.text(1.9, 2.1, "Tn5", fontsize=7, ha="center", color="#e6550d")
for a in np.linspace(0, 2 * np.pi, 9)[:-1]:                                               # surface proteins + antibodies
    x, y = 2.0 + 1.45 * np.cos(a), 1.6 + 1.45 * np.sin(a)
    ax.plot([x, x + 0.25 * np.cos(a)], [y, y + 0.25 * np.sin(a)], color="#31a354", lw=2)
    ax.plot(x + 0.33 * np.cos(a), y + 0.33 * np.sin(a), "v", color="#756bb1", ms=6)
for (x, y) in [(1.1, 1.0), (2.8, 1.1), (1.3, 2.3), (2.7, 2.2)]:                          # mRNAs
    ax.plot([x, x + 0.25], [y, y + 0.05], color="#d62728", lw=2)
texts = [(5.0, 2.65, "RNA", "#d62728", "mRNA molecules captured by poly-T\n→ UMI counts per gene (Module 30)"),
         (5.0, 1.6, "ADT (protein)", "#756bb1", "barcoded antibodies bound to surface proteins\n→ tag counts per protein, plus background"),
         (5.0, 0.55, "ATAC (chromatin)", "#e6550d", "Tn5 cuts open DNA between nucleosomes\n→ open / closed per peak (binary)")]
for x, y, title, col, desc in texts:
    ax.text(x, y + 0.15, title, fontsize=10, color=col, weight="bold", va="bottom")
    ax.text(x, y + 0.1, desc, fontsize=8, va="top")
ax.set_title("One cell, three read-outs: green stalks = surface proteins, purple = antibody with DNA barcode, "
             "grey = nucleosomes", fontsize=9)
plt.show()

In [ ]:
# ── 1.2 The SIMULATED trimodal experiment ──
mo = simulate_multiome(np.random.default_rng(SEED + 1), n_cells=2500, n_genes=800, n_peaks=2000)
rna, adt, atac, obs = mo["rna"], mo["adt"], mo["atac"], mo["obs"]
genes, proteins, peaks = mo["var"].index.to_numpy(str), list(mo["proteins"].index), mo["peaks"]
true_type, t_state = obs.true_type.to_numpy(str), obs.t_state.to_numpy(str)
is_treg, is_primed = obs.is_treg.to_numpy(bool), obs.is_primed.to_numpy(bool)


def fine_state(cell_type, state, treg, primed):
    """The full hidden truth of one cell: its type refined by the states that RNA cannot see."""
    if treg:
        return "Treg"
    if state:
        return f"{cell_type} {state}"
    return "Monocyte (primed)" if primed else cell_type


fine = np.array([fine_state(*x) for x in zip(true_type, t_state, is_treg, is_primed)])
print(f"SIMULATED: {len(obs)} cells | RNA {rna.shape[1]} genes | ADT {adt.shape[1]} proteins | ATAC {atac.shape[1]} peaks")
for name, M in [("RNA", rna), ("ADT", adt), ("ATAC", atac)]:
    print(f"  {name:4s}: {np.mean(M > 0):6.1%} non-zero entries, median total per cell {np.median(M.sum(axis=1)):7.0f}")
print("\nhidden truth — 10 fine states:\n", pd.Series(fine).value_counts().to_string())
print("\npeaks by kind:", peaks.kind.value_counts().to_dict(), "| peaks linked to interferon genes:", int(peaks.is_interferon.sum()))

In [ ]:
# ── 1.3 What the raw data look like ──
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
bins = np.linspace(0, 7.5, 40)
for p, col in [("CD3", CLASS_COLORS[0]), ("CD25", CLASS_COLORS[1]), ("IgG1-isotype", "0.4")]:
    axes[0].hist(np.log1p(adt[:, proteins.index(p)]), bins=bins, histtype="step", lw=2, color=col, label=p)
axes[0].set_yscale("log"); axes[0].set_ylim(0.7, 4e3)
axes[0].set_xlabel("log(1 + ADT count)"); axes[0].set_ylabel("cells (log scale)")
axes[0].legend(fontsize=7, ncol=3, loc="upper center", handlelength=1)
axes[0].set_title("ADT: positive cells far above a background\nthat every cell has (isotype = background only)")
depth = atac.sum(axis=1)
axes[1].hist(depth, bins=50, color="0.6")
axes[1].set_xlabel("open peaks per cell (ATAC depth)"); axes[1].set_ylabel("cells")
axes[1].set_title(f"ATAC: depth varies {np.percentile(depth, 95) / np.percentile(depth, 5):.0f}-fold (5th–95th pct.)")
show = np.argsort(depth)[::250]
axes[2].imshow(atac[np.ix_(show, np.arange(300))], aspect="auto", cmap="Greys", interpolation="nearest")
axes[2].set_xlabel("first 300 peaks"); axes[2].set_ylabel("10 cells, sorted by depth")
axes[2].set_title("ATAC is binary and sparse")
plt.suptitle("SIMULATED raw multimodal data", y=1.03)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The three matrices could hardly be more different. RNA: about 30 % non-zero entries and
roughly a thousand UMIs per cell. ADT: almost every entry is non-zero — each of the 15 proteins gets tens to hundreds
of counts in every cell — because antibodies are abundant and stick everywhere a little. The isotype control (grey) has
no target, yet every cell shows about $e^{2.8} \approx 16$ counts of it: that is the **background**. CD3 (blue) splits
cleanly into negative cells at the background level and positive T cells far above it; CD25 (red) is positive only in
the 45 rare Tregs (the small bump on the right — note the log scale). ATAC: only about 4 % of entries are 1, a typical
cell has about 80 open peaks, and that number varies about four-fold between the 5th and 95th percentile of cells,
mostly because of capture efficiency, not biology.

The hidden truth has ten fine states. RNA was generated *without* any difference between naive and memory T cells,
Tregs and other CD4 T cells, or primed and resting monocytes — so whatever we find about them must come from the other
two modalities.

✏️ **Your turn.** Compute, for every protein, the median ADT count in the cells of the type with the highest level and
in all other cells. Which protein has the smallest positive-to-background ratio? (Hint: `ADT_PANEL` lists the levels.)

---
## 2 · Normalising each modality

> 📍 **Where we are.** Each modality has its own noise model, so each gets its own normalisation and its own
> low-dimensional embedding. Integration (Section 4) then combines the *embeddings*, not the raw counts.

### RNA — the Module 30 recipe

Library-size normalisation and log, the 400 most variable genes, z-scores, PCA with 20 components.

> 🔗 **Builds on** [Module 30](30_single_cell_rnaseq.ipynb), Sections 3–4 (normalisation, HVGs, PCA) and
> [Module 14](14_pca_dimensionality_reduction.ipynb) (PCA by SVD).

In [ ]:
# ── 2.1 RNA: log-normalise, 400 HVGs, z-score, PCA ──
L_rna = log_normalize(rna)
hvg = np.argsort(L_rna.var(axis=0))[::-1][:400]
X_rna = (L_rna[:, hvg] - L_rna[:, hvg].mean(axis=0)) / L_rna[:, hvg].std(axis=0)
Z_rna, _, evr_rna = pca(X_rna, n_components=20)
print(f"RNA embedding: {Z_rna.shape}, variance explained by 20 PCs: {evr_rna.sum():.2f}")

### Proteins — the centred log-ratio (CLR)

An ADT count mixes the true amount of protein with a cell-specific technical factor: bigger cells and droplets that
captured more antibody give higher counts for *every* protein. The standard Seurat workflow for CITE-seq uses the
**centred log-ratio** (CLR) transform from compositional data analysis: compare each protein with the cell's own average.

$$\operatorname{clr}(x)_{cp} = \log(1 + x_{cp}) - \frac{1}{P}\sum_{q=1}^{P}\log(1 + x_{cq}) .$$

*In words:* the log count of protein $p$ in cell $c$ minus the mean log count of all $P$ proteins in that cell — i.e.
the log of the ratio between the count and the cell's geometric mean. A factor that multiplies all counts of a cell
cancels. (Seurat's implementation differs in a detail — it divides by the geometric mean before taking `log1p` — with
the same idea.)

**The catch:** the cell's geometric mean depends on *how many* panel proteins the cell expresses. A B cell expresses
four of our 15 proteins strongly, a CD8 T cell three — so the same background count gets a different CLR value in
different cell types. With hundreds of antibodies this matters less; with small panels it is a real artefact.

### Proteins — background-aware scaling (the DSB idea)

Mulè, Martins & Tsang (2022) proposed **DSB** ("denoised and scaled by background"): (1) express each protein in units
of its own background — DSB uses empty droplets, which contain only ambient antibody — and (2) remove a per-cell
technical component estimated from the isotype controls and the cell's overall background level. Our simplified
**DSB-lite** has no empty droplets, so step 1 uses the negative population of each protein instead:

1. for each protein fit a two-component Gaussian mixture to $\log(1 + x)$ across cells ([Module 13](13_mixture_models_em.ipynb))
   and standardise with the mean $\mu_p$ and SD $\sigma_p$ of the lower ("negative") component:
   $z_{cp} = (\log(1 + x_{cp}) - \mu_p)/\sigma_p$;
2. per cell, average two noisy measures of the technical factor — the mean of the cell's lower half of $z$ values (its
   background level) and its isotype $z$ — into $t_c$, and regress it out of every protein: $\tilde z_{cp} = z_{cp} - \beta_p t_c$.

*In words:* a value of 0 now means "at background" and a value of 5 means "five background SDs above it", for every
protein alike — a scale an immunologist can gate on.

In [ ]:
# ── 2.2 CLR and DSB-lite for the antibody tags ──
def clr_transform(counts):
    """Centred log-ratio per cell: log(1 + x) minus the cell's mean log(1 + x) over all proteins."""
    lx = np.log1p(np.asarray(counts, float))
    return lx - lx.mean(axis=1, keepdims=True)


def dsb_lite(counts, isotype_col, seed=SEED):
    """Simplified DSB (Mulè et al. 2022): background units per protein (negative mixture component), then
    regress out a per-cell technical factor built from the isotype and the cell's background level."""
    lx = np.log1p(np.asarray(counts, float))
    z = np.empty_like(lx)
    for j in range(lx.shape[1]):
        if j == isotype_col:                                    # the isotype is background everywhere
            mu, sd = lx[:, j].mean(), lx[:, j].std()
        else:
            gm = GaussianMixture(2, random_state=seed).fit(lx[:, [j]])
            neg = np.argmin(gm.means_.ravel())
            mu, sd = gm.means_.ravel()[neg], np.sqrt(gm.covariances_.ravel()[neg])
        z[:, j] = (lx[:, j] - mu) / sd
    background = np.sort(z, axis=1)[:, : z.shape[1] // 2].mean(axis=1)    # cell's lower half of proteins
    t = np.column_stack([background, z[:, isotype_col]])
    tech = ((t - t.mean(axis=0)) / t.std(axis=0)).mean(axis=1)
    beta = np.linalg.lstsq(np.column_stack([np.ones(len(tech)), tech]), z, rcond=None)[0][1]
    return z - np.outer(tech, beta), tech


ISO = proteins.index("IgG1-isotype")
adt_clr = clr_transform(adt)
adt_dsb, tech = dsb_lite(adt, ISO)
Z_adt, _, evr_adt = pca(adt_clr, n_components=10)
print(f"ADT embedding (PCA of CLR values): {Z_adt.shape}, variance explained: {evr_adt.sum():.2f}")
print(f"technical factor vs log isotype count: r = {np.corrcoef(tech, np.log1p(adt[:, ISO]))[0, 1]:.2f}")

Because the simulator's antibody panel is known, we can reconstruct the **true expression level** (0–1) of every protein
in every cell and ask three questions of each normalisation, using the cells that truly lack a protein (level 0):

- **technical leak** — do their values still go up and down with the cell's isotype count (a proxy for the technical
  factor)? Ideal: correlation 0.
- **cell-type artefact** — does a negative cell's value depend on its cell type? We report the spread of the
  per-type means, relative to the gap between positive and negative cells. Ideal: 0.
- **is zero the background?** — the mean value of negative cells relative to the same gap. Ideal: 0.

In [ ]:
# ── 2.3 🧪 Ground-truth check of three protein normalisations ──
is_t = np.isin(true_type, ["CD4 T", "CD8 T"])
level = np.zeros(adt.shape)                                  # the simulator's true protein levels (0–1)
for j, (p, (_, by_type)) in enumerate(ADT_PANEL.items()):
    for t, v in by_type.items():
        level[true_type == t, j] = v
level[is_t & (t_state == "naive"), proteins.index("CD45RA")] = 1.0
level[t_state == "memory", proteins.index("CD45RO")] = 1.0
level[is_treg, proteins.index("CD25")] = 1.0

log_iso = np.log1p(adt[:, ISO])
rows = {}
for name, M in [("log(1 + count)", np.log1p(adt)), ("CLR", adt_clr), ("DSB-lite", adt_dsb)]:
    leak, spread, zero, auc = [], [], [], []
    for j in range(len(proteins) - 1):                         # every protein except the isotype
        neg, pos = level[:, j] == 0, level[:, j] >= 0.5
        gap = M[pos, j].mean() - M[neg, j].mean()
        leak.append(np.corrcoef(M[neg, j], log_iso[neg])[0, 1])
        spread.append(pd.Series(M[neg, j]).groupby(true_type[neg]).mean().std() / gap)
        zero.append(abs(M[neg, j].mean()) / gap)
        auc.append(roc_auc_score(pos[pos | neg], M[pos | neg, j]))
    rows[name] = {"technical leak (r)": np.mean(leak), "cell-type artefact": np.mean(spread),
                  "|negatives| / gap": np.mean(zero), "AUROC pos vs neg": np.mean(auc)}
print("🧪 averages over the 14 targeted proteins:")
print(pd.DataFrame(rows).T.round(3).to_string())

In [ ]:
# ── 2.4 One protein, two normalisations: CD8 in every cell type ──
types = sorted(set(true_type))
j = proteins.index("CD8")
fig, axes = plt.subplots(1, 2, figsize=(11, 3.3))
for ax, M, name in [(axes[0], adt_clr, "CLR"), (axes[1], adt_dsb, "DSB-lite")]:
    ax.boxplot([M[true_type == t, j] for t in types], showfliers=False)
    ax.set_xticks(range(1, len(types) + 1)); ax.set_xticklabels(types, fontsize=8)
    ax.axhline(0, color="k", lw=0.8, ls="--"); ax.set_ylabel(f"CD8 ({name})")
    neg_means = [M[(true_type == t), j].mean() for t in types if t != "CD8 T"]
    ax.set_title(f"{name}: negatives sit at {min(neg_means):.2f} to {max(neg_means):.2f}")
plt.suptitle("SIMULATED: only CD8 T cells express CD8 — where do the negatives land?", y=1.02)
plt.tight_layout(); plt.show()

🔍 **What just happened?** All three versions separate positive from negative cells almost perfectly (AUROC ≈ 1): with
hundreds of counts above a background of about 15, any monotone transform does. The differences are in the details
that matter for downstream analysis:

- **Raw log counts** leak the technical factor: negative cells with a high isotype count also have high counts for every
  other protein (r = 0.26), and "negative" sits far from zero (0.82 of the positive–negative gap).
- **CLR** removes the leak (r ≈ −0.06) but introduces the **cell-type artefact**: the negatives of a protein land at
  different values in different cell types (spread 0.045 of the gap, against 0.007 for raw counts — six times more).
  In the box plot, CD8-negative dendritic and NK cells sit visibly higher (means −0.52 and −0.63) than CD8-negative B cells,
  CD4 T cells and monocytes (≈ −0.88), because their geometric mean is lower.
- **DSB-lite** keeps the cell-type spread at the raw level (0.007), puts the negatives at zero for every protein
  ("0 = background"; 0.002 of the gap) and turns the values into background-SD units. It slightly *over*-corrects the
  technical factor (r = −0.16): the isotype is itself a noisy measurement, and its noise enters our estimate $t_c$. DSB
  proper reduces this by combining several isotypes with the cell's background level from a per-cell mixture model.

None of this changes which cells are CD3⁺, but it changes distances between cells — and therefore every neighbour graph
we build. For the embedding we use the CLR values (the most common default); ✏️ below lets you try DSB-lite.

⚠️ **Background is not zero.** Never interpret a raw ADT count as "expressed": every cell has counts for every protein.
Gating needs a background model — empty droplets, isotype controls or the negative population.

✏️ **Your turn.** Replace `adt_clr` by `adt_dsb` in the PCA of cell 2.2 and re-run Section 3. Does anything change in
the ADT row of the table?

### Chromatin — TF-IDF and latent semantic indexing (LSI)

The binary ATAC matrix is processed with a trick from text mining: a cell is a "document", a peak a "word" (Cusanovich
*et al.* 2015, 2018; Signac and ArchR use variants).

$$\mathrm{tf}_{cp} = \frac{x_{cp}}{\sum_{q} x_{cq}}, \qquad \mathrm{idf}_p = \frac{N}{\sum_{c} x_{cp}}, \qquad
w_{cp} = \log\!\big(1 + 10^4\,\mathrm{tf}_{cp}\,\mathrm{idf}_p\big) .$$

*In words:* **term frequency** divides each open peak by the cell's number of open peaks (a depth normalisation, like
library size for RNA); **inverse document frequency** up-weights peaks that are open in few cells (rare peaks are more
informative than peaks open everywhere); the log compresses the result. **LSI** is then simply a truncated SVD of
$W$ — PCA without centring ([Module 14](14_pca_dimensionality_reduction.ipynb)).

⚠️ **The first LSI component is usually depth.** Without centring, the first singular vector captures the average
pattern, whose strength in each cell tracks how many fragments the cell has. Standard practice (Signac, ArchR) is to
check its correlation with depth and drop it. We also L2-normalise each cell's remaining coordinates (an option of
neighbour searches, e.g. `l2.norm` in Seurat's `FindNeighbors`) — the next cell shows why that matters here.

In [ ]:
# ── 2.5 TF-IDF + LSI, and the depth component ──
def tfidf_lsi(B, n_components=31, log=True, seed=SEED):
    """Latent semantic indexing of a binary cell × peak matrix: (log-)TF-IDF, then a truncated SVD (no centring)."""
    B = np.asarray(B, float)
    tf = B / np.maximum(B.sum(axis=1, keepdims=True), 1)
    idf = B.shape[0] / np.maximum(B.sum(axis=0), 1)
    W = np.log1p(1e4 * tf * idf) if log else tf * idf
    U, S, Vt = randomized_svd(W, n_components, random_state=seed)
    return U * S, W


t0 = time.time()
lsi, W_atac = tfidf_lsi(atac)
log_depth = np.log(depth)
r_depth = [np.corrcoef(lsi[:, k], log_depth)[0, 1] for k in range(5)]
print(f"LSI in {time.time() - t0:.1f} s; correlation of components 1–5 with log depth:", np.round(r_depth, 2))
Z_atac = lsi[:, 1:31] / np.linalg.norm(lsi[:, 1:31], axis=1, keepdims=True)    # drop component 1, L2-normalise

In [ ]:
# ── 2.6 🧪 Does depth leak into the neighbourhoods? Three ways to embed the chromatin ──
def depth_leak(Z, k=15):
    """Correlation between a cell's log depth and the mean log depth of its k nearest neighbours."""
    nbr = NearestNeighbors(n_neighbors=k + 1).fit(Z).kneighbors(Z)[1][:, 1:]
    return np.corrcoef(log_depth, log_depth[nbr].mean(axis=1))[0, 1], nbr


def unit_rows(Z):
    return Z / np.linalg.norm(Z, axis=1, keepdims=True)


plain, _ = tfidf_lsi(atac, log=False)
variants = {"log-TF-IDF, components 1–30": lsi[:, :30], "log-TF-IDF, components 2–31 (drop 1)": lsi[:, 1:31],
            "log-TF-IDF, 1–30, L2-normalised": unit_rows(lsi[:, :30]),
            "log-TF-IDF, 2–31, L2-normalised (used)": Z_atac, "plain TF-IDF (no log), 2–31, L2": unit_rows(plain[:, 1:31])}
rows = {}
for name, Z in variants.items():
    leak, nbr = depth_leak(Z)
    rows[name] = {"depth leak (r)": leak, "neighbours of the same true type": np.mean(true_type[nbr] == true_type[:, None])}
print(pd.DataFrame(rows).T.round(3).to_string())

fig, ax = plt.subplots(figsize=(5.5, 3.3))
ax.scatter(depth, lsi[:, 0], s=3, color="0.4")
ax.set_xscale("log"); ax.set_xlabel("open peaks per cell (log scale)"); ax.set_ylabel("LSI component 1")
ax.set_title(f"SIMULATED: LSI 1 tracks depth (r = {r_depth[0]:.2f} with log depth)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** LSI component 1 correlates almost perfectly with log depth (r = 0.95; the sign of a singular
vector is arbitrary), the following components hardly at all (|r| ≤ 0.16). With it, neighbourhoods are depth-matched:
the mean depth of a cell's 15 neighbours predicts its own depth almost perfectly (leak r = 0.96) — a purely technical
similarity. The table shows that two standard steps share the work:

- dropping component 1 *alone* barely helps (0.84), because the magnitude of the remaining coordinates still grows with
  the number of open peaks;
- **L2-normalising** each cell's coordinates — keeping only the *direction* of its vector — is what removes most of the
  leak (0.35 with component 1, **0.25 without it**) and gives the highest fraction of neighbours of the same true type
  (0.87). Dropping the depth component *and* normalising is the combination we use.

The last row is a warning about the "log" in log-TF-IDF: without it, a few rare peaks in shallow cells get enormous
weights ($\mathrm{tf}\cdot\mathrm{idf}$ is largest exactly there) and dominate the embedding. Its depth leak is small,
but only 20 % of a cell's neighbours share its type — with six equally common types, that is chance level.

✏️ **Your turn.** Rerun `tfidf_lsi` on the monocytes only (`atac[true_type == "Monocyte"]`), then with `log=False`, and
correlate the first five components of each with `is_primed[true_type == "Monocyte"]`. One version has a component
that tracks priming almost perfectly, the other has none. Which is which, and why — using the weighting argument above?

---
## 3 · One modality at a time

> 📍 **Where we are.** We have three embeddings — 20 RNA PCs, 10 ADT PCs, 30 ATAC LSI components. Before combining
> them, we ask what each sees *alone*, with the clustering of Module 30 (k-NN graph + Louvain) and the hidden truth.

Two kinds of score:

- the **adjusted Rand index** (ARI, [Module 13](13_mixture_models_em.ipynb)) of the clusters against the 6 true types
  and against the 10 fine states;
- a **neighbourhood AUROC** for each hidden state, computed inside the population where it lives: score every cell by
  the fraction of its 15 neighbours that are in the state (e.g. memory), and compute the AUROC of that score against
  the truth. 0.5 means the state is invisible in this embedding, 1.0 that cells of the state have only each other as
  neighbours. Unlike the ARI it does not depend on whether a clustering happens to cut at the right place.

In [ ]:
# ── 3.1 Helpers: k-NN graph, Louvain, and the two scores ──
def knn_indices(Z, k=15):
    """Indices of each cell's k nearest neighbours (excluding itself)."""
    return NearestNeighbors(n_neighbors=k + 1).fit(Z).kneighbors(Z)[1][:, 1:]


def louvain_clusters(nbr, resolution=1.0):
    """Louvain communities of the k-NN graph given as a neighbour-index matrix (as in Module 30)."""
    g = nx.Graph()
    g.add_nodes_from(range(len(nbr)))
    g.add_edges_from(zip(np.repeat(np.arange(len(nbr)), nbr.shape[1]), nbr.ravel()))
    comms = sorted(nx.community.louvain_communities(g, resolution=resolution, seed=SEED), key=len, reverse=True)
    lab = np.empty(len(nbr), int)
    for c, members in enumerate(comms):
        lab[list(members)] = c
    return lab


HIDDEN = {"memory (T cells)": (is_t, t_state == "memory"), "Treg (CD4 T)": (true_type == "CD4 T", is_treg),
          "primed (monocytes)": (true_type == "Monocyte", is_primed)}


def score_embedding(name, Z=None, nbr=None):
    """Cluster an embedding (or a given neighbour matrix) and score it against the hidden truth."""
    nbr = knn_indices(Z) if nbr is None else nbr
    lab = louvain_clusters(nbr)
    row = {"clusters": lab.max() + 1, "ARI type": adjusted_rand_score(true_type, lab),
           "ARI fine": adjusted_rand_score(fine, lab)}
    for state, (pop, y) in HIDDEN.items():
        row[f"nbr-AUROC {state}"] = roc_auc_score(y[pop], y[nbr[pop]].mean(axis=1))
    return pd.Series(row, name=name), lab

In [ ]:
# ── 3.2 🧪 What does each modality see on its own? ──
t0 = time.time()
results, labels = [], {}
for name, Z in [("RNA", Z_rna), ("ADT", Z_adt), ("ATAC", Z_atac)]:
    row, labels[name] = score_embedding(name, Z)
    results.append(row)
single = pd.DataFrame(results)
print(f"🧪 one modality at a time ({time.time() - t0:.1f} s):")
print(single.round(2).to_string())

In [ ]:
# ── 3.3 Hidden states, seen through each modality (PCA inside each population) ──
t_lab = np.where(is_treg, "Treg", t_state)
m_lab = np.where(is_primed, "primed", "resting")
fig, axes = plt.subplots(2, 3, figsize=(12, 6.2))
for col, (name, Z) in enumerate([("RNA", Z_rna), ("ADT", Z_adt), ("ATAC", Z_atac)]):
    for row, (pop, lab, what) in enumerate([(is_t & (true_type == "CD4 T"), t_lab, "CD4 T cells"),
                                            (true_type == "Monocyte", m_lab, "monocytes")]):
        P2 = pca(Z[pop], n_components=2)[0]
        plot_embedding(P2, lab[pop], ax=axes[row, col], s=5, title=f"{what} in the {name} embedding",
                       legend=(col == 0))
plt.suptitle("SIMULATED: each hidden state is visible in exactly one modality", y=1.0)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The table and the figure tell one story.

- **RNA** recovers the six cell types almost perfectly (ARI 0.99) but is blind to every hidden state: its neighbourhood
  AUROCs are 0.47–0.56, i.e. chance. Inside the CD4 T cells, naive, memory and Treg cells are mixed in the RNA
  embedding (top left).
- **ADT** sees naive vs memory and the Tregs perfectly (AUROC 1.00), and its clusters match the ten fine states best
  (ARI 0.94). Its lower ARI against the six types (0.88) is not an error: it splits the T cells into naive, memory and
  Treg clusters, finer than "type". Its one blind spot is priming (0.53).
- **ATAC** is the only modality that sees the primed monocytes (AUROC 1.00, bottom right) but is the weakest on cell
  types (ARI 0.82): besides splitting the monocytes by priming, its sparse, noisy profiles put a few per cent of the
  cells into the wrong cluster, and it cannot see the T-cell states.

No single modality gets everything. This is the whole case for multimodal assays — and for integration methods that can
use each modality where it is informative.

✏️ **Your turn.** Cross-tabulate `fine` against `labels["ATAC"]` and against `labels["ADT"]`. Where do the ATAC errors
come from — merged types, split types or scattered cells? Which fine states does each modality merge?

### 📄 Research connection — Multimodal single-cell assays

- **What the field does.** CITE-seq (Stoeckius *et al.* 2017) and REAP-seq (Peterson *et al.* 2017) added DNA-barcoded antibodies to droplet
    scRNA-seq; scATAC-seq (Buenrostro *et al.* 2015; Cusanovich *et al.* 2015) brought chromatin accessibility to single
    cells; SHARE-seq (Ma *et al.* 2020) and the commercial 10x Multiome kit measure RNA and chromatin in the same nucleus,
    and trimodal methods (TEA-seq, DOGMA-seq) add proteins. Related assays add DNA methylation, histone marks (scCUT&Tag),
    lineage barcodes or spatial position. Reviews such as Efremova & Teichmann (2020) map the computational problems.
- **What this notebook builds.** A simulated trimodal experiment in which some states are visible in only one modality *by construction*,
    with a normalisation per modality (CLR and a DSB-style background model for proteins, TF-IDF + LSI for chromatin)
    and a per-modality analysis that quantifies what each one sees.
- **Where the frontier is.** Larger antibody panels and protein measurements beyond the surface, higher-throughput combinatorial-indexing
    multiome methods, and joint measurement with spatial position are active areas. A recurring open question is how much
    each extra modality adds for a given biological question and cost; another is how to normalise protein counts without
    empty droplets or isotype controls.
- **Read next.** Stoeckius *et al.* 2017 (*Nature Methods*), *Simultaneous epitope and transcriptome measurement in single cells*; Ma *et al.* 2020 (*Cell*), *Chromatin potential identified by shared single-cell profiling of RNA and chromatin*; Mulè, Martins & Tsang 2022 (*Nature Communications*), the DSB normalisation.

---
## 4 · Integration strategies

> 📍 **Where we are.** Three embeddings, each blind to something. Integration methods are usually sorted by *when* the
> modalities are combined:
>
> - **early** — concatenate the per-modality features or embeddings, then analyse as one data set;
> - **late** — analyse each modality separately and combine the *results* (e.g. clusterings);
> - **intermediate** — learn a joint representation: a shared space (CCA), a joint graph (WNN), or a joint latent
>   variable model (MOFA+, totalVI — Section 7).

### Early integration — and the scaling trap

Concatenation is simple but needs care: our three embeddings live on completely different scales (20 PCs of z-scored
genes, 10 PCs of CLR values, 30 L2-normalised LSI coordinates). Without rescaling, the block with the largest total
variance dominates every distance. A minimal fix is to give each block the same total variance.

In [ ]:
# ── 4.1 Early integration: concatenation with and without balancing the blocks ──
def balanced(Z):
    """Centre an embedding and scale it so that its total variance (mean squared norm) is 1."""
    Z = Z - Z.mean(axis=0)
    return Z / np.sqrt((Z ** 2).sum(axis=1).mean())


blocks_raw = {"RNA": Z_rna, "ADT": Z_adt, "ATAC": lsi[:, 1:31]}
print("total variance of each block before balancing:",
      {k: round(float(((v - v.mean(axis=0)) ** 2).sum(axis=1).mean()), 1) for k, v in blocks_raw.items()})
Z_early_raw = np.hstack(list(blocks_raw.values()))
Z_early = np.hstack([balanced(Z_rna), balanced(Z_adt), balanced(Z_atac)])
integrated = []
for name, Z in [("early, unscaled concatenation", Z_early_raw), ("early, balanced blocks", Z_early)]:
    row, labels[name] = score_embedding(name, Z)
    integrated.append(row)
print(pd.DataFrame(integrated).round(2).to_string())

🔍 **What just happened?** Unscaled, the ATAC block (whose un-normalised LSI coordinates have a total variance of about
360) and the RNA block (69) swamp the ADT block (26): the joint embedding sees priming and cell types, but the Tregs
vanish (AUROC 0.58) and the fine-state ARI drops to 0.86. Giving every block the same total variance fixes it: the
balanced concatenation reaches a fine-state ARI of 0.99 and sees all three hidden states. On this clean simulation,
a carefully scaled early integration is already very good — keep that in mind as a **baseline** for the fancier methods
below.

### Late integration — combining clusterings

The simplest late strategy: cluster each modality separately and give each cell the *combination* of its three cluster
labels. A state found by any modality survives; the price is that disagreements between modalities create many small
groups.

In [ ]:
# ── 4.2 Late integration: the combination of the three single-modality clusterings ──
combo = np.array([f"{a}-{b}-{c}" for a, b, c in zip(labels["RNA"], labels["ADT"], labels["ATAC"])])
sizes = pd.Series(combo).value_counts()
row = pd.Series({"clusters": len(sizes), "ARI type": adjusted_rand_score(true_type, combo),
                 "ARI fine": adjusted_rand_score(fine, combo)}, name="late, label combinations")
integrated.append(row)
print(row.round(2).to_string())
print(f"groups with ≥ 20 cells: {(sizes >= 20).sum()} (holding {sizes[sizes >= 20].sum()} cells); "
      f"groups with < 5 cells: {(sizes < 5).sum()}")

🔍 **What just happened?** The combinations produce 57 groups: ten large ones (2,325 cells) that correspond to the fine
states, and dozens of tiny ones (35 with fewer than five cells) where ATAC (weak on cell types) disagrees with the
other two. The fine-state ARI (0.94) is decent,
the type ARI poor (0.76), because many cells of one type are scattered over several small groups. Late integration
inherits every error of every modality; it is useful mainly when modalities were measured in *different* cells and no
joint representation is possible.

### Intermediate integration I — canonical correlation analysis (CCA)

CCA (Hotelling 1936) finds directions in two data sets that are maximally *correlated* across the same cells. Seurat v3
used it to align different experiments (Stuart *et al.* 2019); here we apply it to two modalities of the same cells.

🧮 **Math refresher — CCA by the SVD.** Let $X$ ($N \times p$) and $Y$ ($N \times q$) be centred, with covariance matrices
$C_{XX}$, $C_{YY}$ and cross-covariance $C_{XY} = X^\top Y/(N-1)$. We look for weight vectors $a$, $b$ that maximise

$$\rho = \operatorname{corr}(Xa, Yb) = \frac{a^\top C_{XY} b}{\sqrt{a^\top C_{XX} a}\,\sqrt{b^\top C_{YY} b}} .$$

*In words:* one score per cell from each modality, chosen so that the two scores agree as well as possible across cells.
Substitute $u = C_{XX}^{1/2} a$ and $v = C_{YY}^{1/2} b$ ("whitening": in the new coordinates each data set has identity
covariance); the denominator becomes $\|u\|\,\|v\|$ and

$$\rho = \frac{u^\top \big(C_{XX}^{-1/2} C_{XY} C_{YY}^{-1/2}\big) v}{\|u\|\,\|v\|} .$$

*In words:* after whitening, CCA is the problem "find the unit vectors that maximise $u^\top M v$" for the matrix
$M = C_{XX}^{-1/2} C_{XY} C_{YY}^{-1/2}$ — whose answer is the top singular vectors of $M$ ([Module 03](03_linear_algebra_for_data.ipynb)).
The singular values are the **canonical correlations** $\rho_1 \ge \rho_2 \ge \dots$, and the weights are
$a_k = C_{XX}^{-1/2} u_k$, $b_k = C_{YY}^{-1/2} v_k$.

> 🔗 **Builds on** [Module 14](14_pca_dimensionality_reduction.ipynb) (PCA maximises *variance* within one data set;
> CCA maximises *correlation* between two) and [Module 18](18_gaussian_graphical_models.ipynb) (covariance and
> precision matrices; $C^{-1/2}$ is the "whitening" matrix).

In [ ]:
# ── 4.3 CCA between RNA and protein, derived from the SVD of the whitened cross-covariance ──
def inv_sqrt(C):
    """C^(-1/2) of a symmetric positive-definite matrix, by eigendecomposition."""
    vals, vecs = np.linalg.eigh(C)
    return vecs @ np.diag(vals ** -0.5) @ vecs.T


def cca_fit(X, Y, k, ridge=1e-3):
    """Canonical variates (N × k for each data set) and canonical correlations; `ridge` stabilises the inverses."""
    X, Y = X - X.mean(axis=0), Y - Y.mean(axis=0)
    n = len(X)
    Cxx_is = inv_sqrt(X.T @ X / (n - 1) + ridge * np.eye(X.shape[1]))
    Cyy_is = inv_sqrt(Y.T @ Y / (n - 1) + ridge * np.eye(Y.shape[1]))
    U, s, Vt = np.linalg.svd(Cxx_is @ (X.T @ Y / (n - 1)) @ Cyy_is)
    return X @ (Cxx_is @ U[:, :k]), Y @ (Cyy_is @ Vt[:k].T), s


cv_rna, cv_adt, rho = cca_fit(Z_rna, adt_clr, k=5)
print("canonical correlations (RNA PCs vs CLR proteins):", np.round(rho, 3))
check = [np.corrcoef(cv_rna[:, k], cv_adt[:, k])[0, 1] for k in range(5)]
print("check — correlation of the first 5 pairs of canonical variates:", np.round(check, 3))
Z_cca = np.hstack([cv_rna / cv_rna.std(axis=0), cv_adt / cv_adt.std(axis=0)])
row, labels["CCA (5 shared dims)"] = score_embedding("intermediate: CCA (5 shared dims)", Z_cca)
integrated.append(row)
print(row.round(2).to_string())

In [ ]:
# ── 4.4 What the canonical variates capture ──
fig, axes = plt.subplots(1, 2, figsize=(11, 3.3), gridspec_kw={"width_ratios": [1, 1.3]})
axes[0].bar(np.arange(1, len(rho) + 1), rho, color=["0.3"] * 5 + ["0.75"] * (len(rho) - 5))
axes[0].set_xlabel("canonical pair"); axes[0].set_ylabel("canonical correlation ρ")
axes[0].set_title("5 strongly shared directions, then noise")
mem = t_state == "memory"
r_mem = [abs(np.corrcoef(cv_adt[is_t, k], mem[is_t])[0, 1]) for k in range(5)]
r_type = [max(abs(np.corrcoef(cv_adt[:, k], true_type == t)[0, 1]) for t in types) for k in range(5)]
x = np.arange(1, 6)
axes[1].bar(x - 0.18, r_type, 0.36, color=CLASS_COLORS[0], label="best |r| with a cell-type indicator")
axes[1].bar(x + 0.18, r_mem, 0.36, color=CLASS_COLORS[1], label="|r| with memory (within T cells)")
axes[1].set_xlabel("canonical variate (protein side)"); axes[1].set_ylabel("|correlation|"); axes[1].legend(fontsize=7)
axes[1].set_title("the shared directions encode cell types, not T-cell memory")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The first five canonical correlations are 0.79–0.96, the rest about 0.1: RNA and protein
share exactly five strong directions — with six cell types, five directions suffice to separate them. Clustering in
this shared space recovers the cell types perfectly (ARI 1.00) but **loses every hidden state** (neighbourhood AUROCs
0.46–0.59): the naive/memory difference lives in the proteins only, so it is *uncorrelated* with RNA and CCA, which
maximises correlation, throws it away by design. The right panel confirms it: each canonical variate is strongly tied to
some cell type (|r| 0.63–0.84) and at most weakly to memory (|r| ≤ 0.26). (The 15th canonical correlation is exactly 0:
CLR values of a cell sum to zero, so the 15 proteins span only 14 dimensions.)

⚠️ **CCA keeps what modalities agree on.** That is exactly right for aligning two *experiments* that measured the same
modality (the batch problem of [Module 31](31_batch_integration.ipynb)) and exactly wrong for discovering what only one
modality can see. Choosing an integration method is choosing which variation you consider signal.

### Intermediate integration II — weighted nearest neighbours (WNN-lite)

Hao *et al.* (2021) asked a per-cell question: *for this cell, which modality describes its neighbourhood better?*
Their weighted-nearest-neighbour procedure, simplified:

1. In each modality $m$, find each cell's $k$ nearest neighbours.
2. **Within-modality prediction:** predict cell $i$'s embedding in modality $m$ by the average of its neighbours *in
   $m$*. **Cross-modality prediction:** predict it by the average (in $m$) of its neighbours *in the other modalities*.
3. Turn prediction errors $d$ into affinities with a cell-specific kernel,
   $\theta_m(d) = \exp\!\big(-\max(d - d_{i,1}, 0)/(\sigma_i - d_{i,1})\big)$, where $d_{i,1}$ is the distance to the nearest
   neighbour and $\sigma_i$ the mean distance to the $k$ neighbours (a local bandwidth).
4. The ratio $s_{im} = \theta_m(\text{within}) / (\theta_m(\text{cross}) + \epsilon)$ is large when modality $m$'s own
   neighbours predict the cell much better than the other modalities' neighbours do — when $m$ sees something the others
   miss. The **modality weights** are a softmax over modalities, $w_{im} = e^{\log s_{im}} / \sum_{m'} e^{\log s_{im'}}$.
5. The joint similarity of cells $i$ and $j$ is $\sum_m w_{im}\,\theta_m(d_m(i, j))$; each cell keeps the 15 candidates
   (from the union of its per-modality neighbour lists) with the highest joint similarity — the **WNN graph**.

*In words:* every cell votes, with weights it has learned from its own neighbourhood, on which modality to trust. (Hao
*et al.* apply the softmax to the ratios themselves; with three modalities and very large ratios we use their logarithm
so that the weights do not saturate at 0 and 1 — one of several simplifications.)

In [ ]:
# ── 4.5 WNN-lite: per-cell modality weights and a joint neighbour graph ──
def wnn_lite(embeddings, k=20, k_out=15, eps=1e-4):
    """Simplified weighted-nearest-neighbour integration (after Hao et al. 2021) for any number of modalities.
    Returns the per-cell modality weights (N × M) and the joint neighbour indices (N × k_out)."""
    names, n = list(embeddings), len(next(iter(embeddings.values())))
    fit = {m: NearestNeighbors(n_neighbors=k + 1).fit(Z).kneighbors(Z) for m, Z in embeddings.items()}
    idx = {m: fit[m][1][:, 1:] for m in names}
    d_first = {m: fit[m][0][:, 1] for m in names}                   # distance to the nearest neighbour
    sigma = {m: fit[m][0][:, 1:].mean(axis=1) for m in names}       # local bandwidth

    def affinity(m, d):                                             # d: (n,) or (n, candidates)
        d1, s = (d_first[m], sigma[m]) if d.ndim == 1 else (d_first[m][:, None], sigma[m][:, None])
        return np.exp(-np.maximum(d - d1, 0) / (s - d1))

    log_ratio = []
    for m in names:
        Z = embeddings[m]
        within = np.linalg.norm(Z - Z[idx[m]].mean(axis=1), axis=1)
        cross = np.mean([np.linalg.norm(Z - Z[idx[o]].mean(axis=1), axis=1) for o in names if o != m], axis=0)
        log_ratio.append(np.log(affinity(m, within) / (affinity(m, cross) + eps)))
    S = np.column_stack(log_ratio)
    w = np.exp(S - S.max(axis=1, keepdims=True)); w /= w.sum(axis=1, keepdims=True)
    cand = np.hstack([idx[m] for m in names])                       # candidate neighbours from every modality
    joint = sum(w[:, [a]] * affinity(m, np.linalg.norm(embeddings[m][cand] - embeddings[m][:, None], axis=2))
                for a, m in enumerate(names))
    out = np.empty((n, k_out), int)
    for i in range(n):
        c, first = np.unique(cand[i], return_index=True)            # drop duplicate candidates
        out[i] = c[np.argsort(-joint[i, first])[:k_out]]
    return w, out


t0 = time.time()
wnn_weights, wnn_nbr = wnn_lite({"RNA": Z_rna, "ADT": Z_adt, "ATAC": Z_atac})
row, labels["WNN-lite"] = score_embedding("intermediate: WNN-lite", nbr=wnn_nbr)
integrated.append(row)
print(f"WNN-lite in {time.time() - t0:.1f} s")
print(row.round(2).to_string())

In [ ]:
# ── 4.6 Which modality does each cell trust? ──
wdf = pd.DataFrame(wnn_weights, columns=["RNA", "ADT", "ATAC"])
mean_w = wdf.groupby(fine).mean().loc[sorted(set(fine))]
fig, ax = plt.subplots(figsize=(8, 3.4))
im = ax.imshow(mean_w.T.values, cmap="Blues", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(mean_w))); ax.set_xticklabels(mean_w.index, rotation=35, ha="right", fontsize=8)
ax.set_yticks(range(3)); ax.set_yticklabels(mean_w.columns)
for (r, c), v in np.ndenumerate(mean_w.T.values):
    ax.text(c, r, f"{v:.2f}", ha="center", va="center", fontsize=7, color="white" if v > 0.6 else "black")
plt.colorbar(im, ax=ax, shrink=0.8, label="mean WNN weight")
ax.set_title("SIMULATED: WNN-lite weights by hidden fine state")
plt.tight_layout(); plt.show()

In [ ]:
# ── 4.7 🧪 All strategies against the full truth ──
summary = pd.concat([single, pd.DataFrame(integrated)])
print(summary.round(2).to_string(na_rep="—"))
fig, ax = plt.subplots(figsize=(9, 3.2))
colours = ["0.7"] * len(single) + [CLASS_COLORS[0]] * (len(summary) - len(single))
ax.barh(summary.index[::-1], summary["ARI fine"][::-1], color=colours[::-1])
ax.set_xlabel("ARI against the 10 hidden fine states"); ax.set_xlim(0.5, 1.0)
ax.set_title("SIMULATED: single modalities (grey) vs integration strategies (blue)")
plt.tight_layout(); plt.show()

In [ ]:
# ── 4.8 Where do the two best strategies differ? Contingency tables (rows: hidden fine state) ──
for name in ["WNN-lite", "early, balanced blocks"]:
    tab = pd.crosstab(pd.Series(fine, name="fine state"), pd.Series(labels[name], name=f"{name} cluster"))
    print(f"\n{name}:\n{tab.to_string()}")

🔍 **What just happened?** The weights are biologically sensible. In every T-cell state (and Tregs) WNN-lite puts about
0.9 of the weight on protein — the modality whose own neighbourhoods predict a T cell's embedding far better than the
other modalities can. **Primed monocytes** are the only cells that lean on chromatin (ATAC weight ≈ 0.4, against ≈ 0.15
for resting monocytes). Other types spread their weight, with protein still the largest share: the 15-antibody panel
describes them cleanly, and with three modalities a cell's "cross" prediction averages two others.

In the summary table WNN-lite sees every hidden state (AUROC 0.94–1.00) and matches the best single modality on the fine
states (ARI 0.94). Here it does **not** beat the balanced early concatenation (0.99): the contingency tables of 4.8
show that WNN-lite gives the Tregs two clusters of their own (23 and 21 cells) but leaves all 152 primed monocytes in one
cluster with the 281 resting ones, while the concatenation does the opposite — it separates the primed monocytes (151
of 152) and spreads the Tregs over the naive and memory CD4 T clusters. There are three times more primed monocytes than
Tregs, so the ARI favours the concatenation. On a clean
simulation where all modalities are informative everywhere, a well-scaled concatenation is hard to beat. WNN's
selling points on real data are robustness when one modality is uninformative or low-quality for some cells (a fixed
global weighting then lets that noise into every distance) and the interpretable per-cell weights. Benchmarks of
multimodal integration (e.g. the NeurIPS 2021 challenge, Lance *et al.* 2022) likewise find no method that wins
everywhere.

✏️ **Your turn.** (1) In the tables of 4.8, why might WNN-lite merge the primed monocytes although their ATAC weight is
0.4? (Hint: what do their protein and RNA neighbourhoods look like, and how much weight do those still get?) (2) Simulate a low-quality protein measurement in 40 % of cells — replace their ADT
counts with `rng.binomial(adt[bad], 0.01)` — and recompute both. Which method suffers more?

### 📄 Research connection — Multimodal integration methods

- **What the field does.** Seurat's weighted-nearest-neighbour analysis (Hao *et al.* 2021) learns per-cell modality weights and builds a
    joint graph; CCA (Hotelling 1936) underlies Seurat v3's anchors for aligning data sets (Stuart *et al.* 2019);
    MOFA+ (Argelaguet *et al.* 2020) is a group factor analysis with shared and modality-specific factors; totalVI
    (Gayoso *et al.* 2021) and MultiVI are multimodal variational autoencoders with modality-specific likelihoods. The
    NeurIPS 2021 multimodal single-cell challenge (Lance *et al.* 2022) compared methods for modality prediction,
    matching and joint embedding on bone-marrow CITE-seq and multiome data.
- **What this notebook builds.** Early (balanced and unbalanced concatenation), late (combining clusterings) and intermediate integration
    (CCA derived from the SVD; a simplified WNN for three modalities), all scored against ten hidden fine states.
    We saw CCA discard modality-specific states by design, the scaling trap of concatenation, and per-cell WNN weights
    that point to the informative modality.
- **Where the frontier is.** **Mosaic integration** — combining data sets in which different cells were measured with different
    subsets of modalities (e.g. StabMap and Seurat's bridge integration, both 2024) — is a major direction, as are
    multimodal reference atlases, scalable multimodal VAEs and principled ways to decide which modality to trust for
    which question. As of 2025 there is no consensus best method; strong simple baselines remain important.
- **Read next.** Hao *et al.* 2021 (*Cell*), *Integrated analysis of multimodal single-cell data*; Argelaguet *et al.* 2020 (*Genome Biology*), *MOFA+*; Lance *et al.* 2022 (NeurIPS 2021 Competitions and Demonstrations Track, PMLR 176), *Multimodal single cell data integration challenge: results and lessons learned*.

---
## 5 · Cross-modal prediction

> 📍 **Where we are.** If one modality could be predicted from another, we would not need to measure it. Predicting
> proteins from RNA is a standard task (it was one of the three NeurIPS 2021 challenge tasks) — and a clean way to
> quantify what protein adds.

We hold out 30 % of the cells and predict the 14 CLR protein values from RNA in two ways:

- **ridge regression** on all 800 standardised log-expression values ([Module 07](07_ridge_lasso_sparse_regression.ipynb)),
  with the penalty chosen on a validation split;
- **k-NN regression** on the 20 RNA PCs: average the protein values of the 15 nearest training cells ([Module 08](08_knn_naive_bayes.ipynb)).

In [ ]:
# ── 5.1 Predict every protein from RNA: ridge and k-NN regression, R² on held-out cells ──
rng = np.random.default_rng(SEED + 5)
split = rng.random(len(obs))
train, valid, test = split < 0.5, (split >= 0.5) & (split < 0.7), split >= 0.7
X_all = (L_rna - L_rna.mean(axis=0)) / (L_rna.std(axis=0) + 1e-8)
Y_prot = adt_clr[:, :ISO]                                   # 14 targeted proteins


def r_squared(y, yhat):
    return 1 - ((y - yhat) ** 2).sum(axis=0) / ((y - y.mean(axis=0)) ** 2).sum(axis=0)


val_r2 = {a: r_squared(Y_prot[valid], Ridge(alpha=a).fit(X_all[train], Y_prot[train]).predict(X_all[valid])).mean()
          for a in [10, 100, 1000, 10000]}
best_alpha = max(val_r2, key=val_r2.get)
fit_tr = train | valid
ridge = Ridge(alpha=best_alpha).fit(X_all[fit_tr], Y_prot[fit_tr])
knn_reg = KNeighborsRegressor(n_neighbors=15).fit(Z_rna[fit_tr], Y_prot[fit_tr])
pred_ridge, pred_knn = ridge.predict(X_all[test]), knn_reg.predict(Z_rna[test])
pred_table = pd.DataFrame({"encoding gene measured?": [ADT_PANEL[p][0] in set(genes) for p in proteins[:ISO]],
                           "ridge R²": r_squared(Y_prot[test], pred_ridge), "k-NN R²": r_squared(Y_prot[test], pred_knn)},
                          index=proteins[:ISO])
print("validation R² by ridge penalty:", {a: round(float(v), 3) for a, v in val_r2.items()}, "→ α =", best_alpha)
print("🧪 held-out R² per protein (CLR scale):")
print(pred_table.round(2).to_string())
print(f"mean over the panel: ridge {pred_table['ridge R²'].mean():.2f}, k-NN {pred_table['k-NN R²'].mean():.2f}")

In [ ]:
# ── 5.2 Predicted vs measured for three proteins ──
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
lab_test = np.where(is_treg, "Treg", np.where(t_state != "", "T " + t_state, true_type))[test]
for ax, p in zip(axes, ["CD14", "CD45RO", "CD25"]):
    j = proteins.index(p)
    for k, grp in enumerate(["Monocyte", "T naive", "T memory", "Treg"]):
        m = lab_test == grp
        ax.scatter(Y_prot[test][m, j], pred_knn[m, j], s=6, color=CLASS_COLORS[k], label=grp)
    m = ~np.isin(lab_test, ["Monocyte", "T naive", "T memory", "Treg"])
    ax.scatter(Y_prot[test][m, j], pred_knn[m, j], s=4, color="0.75", label="other types", zorder=0)
    lim = [Y_prot[:, j].min(), Y_prot[:, j].max()]
    ax.plot(lim, lim, "k--", lw=0.8)
    ax.set_xlabel(f"measured {p} (CLR)"); ax.set_ylabel(f"predicted from RNA")
    ax.set_title(f"{p}: k-NN R² = {pred_table.loc[p, 'k-NN R²']:.2f}")
axes[0].legend(fontsize=7, markerscale=2)
plt.suptitle("SIMULATED held-out cells: what RNA can and cannot predict", y=1.02)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The validation split chose a strong ridge penalty (α = 1000; with 800 correlated genes and
a few hundred cells per type, shrinkage pays — Module 07). Proteins that mark **cell types** are predicted well (k-NN
R² 0.84–0.91 for CD3, CD4, CD8, CD14, CD19, CD20, CD11c, HLA-DR …), *even when the encoding gene is not among the measured genes*
(CD56, CD16, CD11c, HLA-DR): RNA identifies the cell type, and the cell type implies the protein. k-NN regression on the
RNA PCs beats ridge on the raw genes here, because protein level is essentially a function of cell type, which a local
average captures directly.

Three proteins resist: **CD45RA** (R² ≈ 0.5 — predictable only to the extent that B and NK cells always carry it),
**CD45RO** (≈ 0.3) and **CD25** (≈ 0). The scatter plots show why: naive and memory T cells (red, green) get the *same*
predicted CD45RO, Tregs (purple) the same CD25 as everyone else. Their RNA is identical — in the simulation by construction; in
real cells because CD45RA/RO are splice isoforms of one gene and CD25 transcripts are sparse. **That is the case for
measuring proteins.** A practical corollary: models that "impute" proteins from RNA can only return what the RNA
already implies.

⚠️ **R² averages hide the interesting part.** A mean R² over the panel (0.65 for ridge, 0.74 for k-NN here) looks good;
the proteins that matter most for fine states are the ones it fails on.

### Gene activity: from chromatin to a gene-level score

Chromatin can be summarised per gene by a **gene activity score**: the number of open peaks linked to the gene (in
real data: peaks in the gene body and promoter, often weighted by distance — Signac's `GeneActivity`, ArchR's gene
scores). Comparing gene activity with RNA asks whether chromatin and transcription agree.

In [ ]:
# ── 5.3 Gene activity from chromatin vs RNA: the primed monocytes ──
linked = peaks.linked_gene.to_numpy(str)
gene_pos = {g: i for i, g in enumerate(genes)}
link_matrix = np.zeros((atac.shape[1], len(genes)))       # peaks × genes, 1 if the peak is linked to the gene
for p, g in enumerate(linked):
    if g:
        link_matrix[p, gene_pos[g]] = 1
activity = atac @ link_matrix                              # open linked peaks per gene and cell
activity_n = np.log1p(1e4 * activity / activity.sum(axis=1, keepdims=True))
ifn = mo["var"].is_interferon.to_numpy(bool)
mono = true_type == "Monocyte"
rows = {}
for name, M in [("gene activity (ATAC)", activity_n), ("RNA (log-normalised)", L_rna)]:
    rows[name] = {"primed monocytes": M[mono & is_primed][:, ifn].mean(), "resting monocytes": M[mono & ~is_primed][:, ifn].mean()}
print("🧪 mean over the 10 interferon genes:")
print(pd.DataFrame(rows).T.round(2).to_string())

ifn_genes = genes[ifn]
fig, ax = plt.subplots(figsize=(9, 3.0))
x = np.arange(len(ifn_genes))
for k, (name, M) in enumerate([("gene activity (ATAC)", activity_n), ("RNA", L_rna)]):
    diff = M[mono & is_primed][:, ifn].mean(axis=0) - M[mono & ~is_primed][:, ifn].mean(axis=0)
    ax.bar(x + 0.2 * (2 * k - 1), diff, 0.4, color=CLASS_COLORS[k], label=name)
ax.set_xticks(x); ax.set_xticklabels(ifn_genes, fontsize=8); ax.axhline(0, color="k", lw=0.8)
ax.set_ylabel("primed − resting (log scale)"); ax.set_ylim(-0.4, 5.2)
ax.legend(fontsize=8, ncol=2, loc="upper center")
ax.set_title("SIMULATED: chromatin at interferon genes opens before any transcript appears")
plt.tight_layout(); plt.show()

🔍 **What just happened?** In primed monocytes the interferon genes' linked peaks are much more often open — gene
activity is about 3.7 log units higher than in resting monocytes — while their RNA is unchanged (0.21 vs 0.19). That is
the signature Ma *et al.* (2020) called **chromatin potential**: regulatory regions that open *ahead* of transcription,
which, in their data, pointed to the fate a cell was about to adopt. In real data, such discordances are also produced
by technical differences between modalities, so they are hypotheses to test (for instance by stimulating the cells and
checking which respond faster).

✏️ **Your turn.** Compute the cell-level correlation between `activity_n[:, j]` and `L_rna[:, j]` for a strongly
type-specific gene (`CD14`, `MS4A1`) and for an anonymous gene such as `Gene0100`. Why is the correlation near zero for
most genes even though peaks open "where the gene is active"? (Hint: in the simulator a peak opens only where its gene
is *strongly* above its average across types.)

---
## 6 · Enhancer–gene links

> 📍 **Where we are.** Measuring RNA and chromatin in the same cells invites a regulatory question: which enhancer
> controls which gene? Multiome methods answer it with **peak–gene correlation**: if a peak is open exactly in the cells
> where a gene is expressed, it is a candidate regulator (Ma *et al.* 2020; Signac's `LinkPeaks`; ArchR's peak-to-gene
> links). Our simulator records the true target gene of every peak, so we can grade the links.

**Metacells.** A single cell's ATAC profile is mostly zeros and its RNA is noisy, so correlations between one peak and
one gene across single cells are weak. Aggregating small groups of similar cells — **metacells** — averages that noise
away. ArchR samples seed cells and aggregates each with its nearest neighbours; we do the same with 200 seeds × 30
neighbours in the balanced joint embedding of Section 4.

In [ ]:
# ── 6.1 Metacells and peak–gene correlations ──
rng = np.random.default_rng(SEED + 6)
seeds = rng.choice(len(obs), 200, replace=False)
groups = NearestNeighbors(n_neighbors=30).fit(Z_early).kneighbors(Z_early[seeds])[1]
meta_rna = np.array([L_rna[g].mean(axis=0) for g in groups])            # 200 metacells × genes
meta_atac = np.array([atac[g].mean(axis=0) for g in groups])            # 200 metacells × peaks (fraction open)
meta_type = np.array([pd.Series(true_type[g]).mode()[0] for g in groups])
print(f"metacell purity (cells of the majority type): {np.mean([np.mean(true_type[g] == t) for g, t in zip(groups, meta_type)]):.3f}")


def zscore_cols(A):
    A = A - A.mean(axis=0)
    sd = A.std(axis=0)
    return A / np.where(sd > 0, sd, 1)


def corr_matrix(A, B):
    """Pearson correlation of every column of A with every column of B (same rows)."""
    return zscore_cols(A).T @ zscore_cols(B) / len(A)


R_meta = corr_matrix(meta_rna, meta_atac)                               # genes × peaks
R_cell = corr_matrix(L_rna, atac.astype(float))                         # the same with single cells
truth = link_matrix.T.astype(bool)                                      # genes × peaks: the true links
type_means = pd.DataFrame(L_rna).groupby(true_type).mean()
variable = type_means.std(axis=0).to_numpy() > np.quantile(type_means.std(axis=0), 0.75)   # type-variable genes
print(f"{truth.sum()} true links; {variable.sum()} type-variable genes carry {truth[variable].sum()} of them")

In [ ]:
# ── 6.2 🧪 How well does correlation find the true links? ──
rows = {}
for name, R, gsel in [("single cells, all genes", R_cell, slice(None)), ("metacells, all genes", R_meta, slice(None)),
                      ("single cells, variable genes", R_cell, variable), ("metacells, variable genes", R_meta, variable)]:
    y, s = truth[gsel].ravel(), R[gsel].ravel()
    rows[name] = {"pairs": len(y), "AUROC": roc_auc_score(y, s), "average precision": average_precision_score(y, s),
                  "base rate": y.mean()}
print(pd.DataFrame(rows).T.round(4).to_string())
j = gene_pos["CD14"]
top = np.argsort(R_meta[j])[::-1][:8]
print("\nCD14 — the 8 peaks most correlated with it across metacells, and the gene each one truly regulates:")
print("  ", [(str(linked[p]) or "none", round(float(R_meta[j, p]), 2)) for p in top])
print("   true CD14 peaks:", [(int(p), round(float(R_meta[j, p]), 2)) for p in np.flatnonzero(linked == "CD14")])

🔍 **What just happened?** Over all 1.6 million gene–peak pairs the correlation barely ranks true links above false ones
(AUROC ≈ 0.59): most genes do not vary between cell types, their peaks never open, and there is nothing to correlate.
Restricting to the 200 most type-variable genes (as real pipelines restrict to variable genes and peaks) raises the
AUROC to 0.76 with metacells (0.74 with single cells). Yet the **average precision is about 0.02** — thirty times the base
rate, but it means that ~98 % of top-ranked "links" are false.

The CD14 example shows why. Among the eight peaks most correlated with *CD14* (r = 0.92–0.94) is CD14's own peak
(0.93, in third place) — but the other seven regulate *S100A8*, *S100A9* and anonymous monocyte genes, and are
correlated just as strongly. Every monocyte-specific peak
correlates with every monocyte-specific gene, because **cell type is a common cause of both**: it opens the peaks of all
monocyte genes and switches on all monocyte transcripts. In the language of [Module 16](16_bayesian_networks_independence.ipynb),
the correlation between a peak and a gene is not a direct dependence; it is the open path peak ← cell type → gene.

Can we condition on the cell type? Let us correlate *within* types: subtract each type's mean from its metacells first.

In [ ]:
# ── 6.3 Conditioning on cell type: within-type correlation ──
def remove_type_means(A, groups_of_rows):
    A = A.copy()
    for t in np.unique(groups_of_rows):
        A[groups_of_rows == t] -= A[groups_of_rows == t].mean(axis=0)
    return A


R_within = corr_matrix(remove_type_means(meta_rna, meta_type), remove_type_means(meta_atac, meta_type))
y = truth[variable].ravel()
print(f"🧪 variable genes — AUROC between types (6.2): {roc_auc_score(y, R_meta[variable].ravel()):.3f}; "
      f"within types: {roc_auc_score(y, R_within[variable].ravel()):.3f}")

🔍 **What just happened?** Within cell types the AUROC drops to about 0.5: conditioning on the cell type removed the
confounding — and *all* of the signal with it. In our simulator, a peak's accessibility depends only on the cell type
(and on priming), never on a cell's individual expression level, so after accounting for type nothing links a peak to
its gene. In real cells there is some within-type co-variation (continuous states, differentiation trajectories — the
variation SHARE-seq exploited), but it is weak, and it is precisely what correlation-based linking needs. Real pipelines
therefore lean on a **prior**: enhancers usually act on genes nearby on the chromosome. Signac and ArchR only test
peak–gene pairs within a window of the gene (hundreds of kilobases), and compare each correlation with background peaks
matched for accessibility and GC content.

Our simulator has no genome coordinates, so we add a toy genome: genes are placed along a 120-Mb "chromosome", promoter
peaks within a few kilobases of their gene, enhancer peaks 5–150 kb away, background peaks anywhere. This is an
**assumption we impose**, mirroring the typical (but not universal) proximity of enhancers to their targets.

In [ ]:
# ── 6.4 A toy genome and a distance window ──
def toy_genome(rng, n_genes, linked, kind, genome_length=120e6):
    """Positions for genes and peaks: promoters ~1 kb from their gene, enhancers 5–150 kb away, background anywhere."""
    gpos = rng.uniform(0, genome_length, n_genes)
    anchor = np.array([gpos[gene_pos[g]] if g else 0.0 for g in linked])
    offset = np.where(kind == "promoter", rng.normal(0, 1e3, len(kind)),
                      rng.choice([-1, 1], len(kind)) * rng.uniform(5e3, 150e3, len(kind)))
    return gpos, np.where(linked != "", anchor + offset, rng.uniform(0, genome_length, len(kind)))


gpos, ppos = toy_genome(np.random.default_rng(SEED + 7), len(genes), linked, peaks.kind.to_numpy(str))
dist = np.abs(gpos[:, None] - ppos[None, :])                          # genes × peaks, base pairs
rows, curves = {}, {}
for name, mask in [("all pairs (variable genes)", np.ones_like(truth) & variable[:, None]),
                   ("within ±200 kb (variable genes)", (dist < 200e3) & variable[:, None])]:
    y, s = truth[mask], R_meta[mask]
    rows[name] = {"candidate pairs": int(mask.sum()), "true links kept": int(y.sum()),
                  "base rate": y.mean(), "average precision": average_precision_score(y, s)}
    curves[name] = precision_recall_curve(y, s)
print(pd.DataFrame(rows).T.round(3).to_string())
fig, ax = plt.subplots(figsize=(6, 3.4))
for k, (name, (prec, rec, _)) in enumerate(curves.items()):
    ax.plot(rec, prec, color=CLASS_COLORS[k], label=name)
ax.set_xlabel("recall of true links"); ax.set_ylabel("precision"); ax.set_yscale("log"); ax.legend(fontsize=7)
ax.set_title("SIMULATED peak–gene links: correlation alone vs correlation + distance")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The window cuts the candidates from about 400,000 pairs to about 1,800, keeps every true link
of the variable genes (by construction of our toy genome) and raises the average precision from 0.024 to about
0.57 (the base rate inside the window is 0.18). The correlation is the same; what changed is the set of hypotheses it
has to choose among. Inside the window it
mostly has to decide between a handful of nearby peaks — and between a gene's own peak and that of a co-expressed
neighbour, it still cannot.

⚠️ **Correlation-based links are hypotheses.** In real genomes the nearest gene is often, but far from always, the target;
some enhancers act over hundreds of kilobases or skip genes. CRISPR perturbation of enhancers (Fulco *et al.* 2019,
who also proposed the activity-by-contact model) provides the causal test — a theme [Module 41](41_perturbation_causal_inference.ipynb)
develops.

✏️ **Your turn.** Try windows of 50 kb and 500 kb. How do the number of true links kept and the average precision
change? What would happen in a real genome, where some true enhancers lie 300 kb away?

### 📄 Research connection — Gene regulation from single-cell multiome data

- **What the field does.** Joint RNA + chromatin profiling turned enhancer–gene linking into a single-cell problem: SHARE-seq (Ma *et al.*
    2020) correlated peaks with genes across cells and defined domains of regulatory chromatin and chromatin potential;
    Signac (Stuart *et al.* 2021) and ArchR (Granja *et al.* 2021) provide peak calling, LSI, gene activity scores, motif
    enrichment and peak-to-gene links for large data sets; gene-regulatory-network methods (e.g. SCENIC+) combine
    accessibility with transcription-factor motifs.
- **What this notebook builds.** Gene activity scores, a chromatin-before-transcription signature, metacells by neighbourhood aggregation and
    peak–gene correlation graded against the true links — with the confounding by cell type made explicit and the effect
    of a genomic-distance prior quantified.
- **Where the frontier is.** Distinguishing causal enhancer–gene links from co-accessibility and co-expression remains hard: approaches
    combine 3-D contact data (activity-by-contact), sequence models ([Module 36](36_regulatory_genomics_deep_learning.ipynb))
    and CRISPR perturbation screens of enhancers. Benchmarks against perturbation data show that all current predictors
    leave much room for improvement.
- **Read next.** Granja *et al.* 2021 (*Nature Genetics*), *ArchR*; Stuart *et al.* 2021 (*Nature Methods*), *Single-cell chromatin state analysis with Signac*; Fulco *et al.* 2019 (*Nature Genetics*), *Activity-by-contact model of enhancer–promoter regulation from thousands of CRISPR perturbations*.

---
## 7 · Joint latent-variable models

> 📍 **Where we are.** CCA found what RNA and protein share; WNN chose per cell. Latent-variable models go further:
> they explain all modalities with one set of latent factors, some shared and some specific to one modality.

**MOFA+** (Argelaguet *et al.* 2018, 2020) is a **group factor analysis**: each modality $m$ is modelled as
$X^{(m)} \approx Z\,W^{(m)\top} + \text{noise}$, with one factor matrix $Z$ (cells × factors) for all modalities and
modality-specific loadings $W^{(m)}$. Sparsity priors on whole factor–modality blocks let a factor be active in all
modalities (shared) or in only one (specific); the model is fitted by variational inference, and its signature output
is a heat map of **variance explained per factor and modality**.

**totalVI** (Gayoso *et al.* 2021) and **MultiVI** (Ashuach *et al.* 2023) are multimodal **VAEs**: one encoder
maps all modalities of a cell to a shared latent vector, and modality-specific decoders use appropriate likelihoods —
negative binomial for RNA, a mixture of a background and a foreground negative binomial for proteins in totalVI (a
probabilistic version of the background model of Section 2), Bernoulli-like for accessibility in MultiVI.

> 🔗 **Builds on** [Module 14](14_pca_dimensionality_reduction.ipynb) (factor analysis and PCA are linear latent-variable
> models) and [Module 25](25_variational_inference_vae.ipynb) (the ELBO, amortised inference, the mini-scVI). A
> multimodal VAE is the mini-scVI with one extra decoder per modality.

We stay linear and implement two small pieces of the idea:

1. a **weighted joint PCA** — the classical *multiple factor analysis*: divide each modality's block by its first
   singular value, so that no modality dominates just because it has more features or larger numbers, concatenate and
   run PCA; then measure the variance each factor explains in each modality;
2. a **shared-plus-private decomposition**: take the five shared CCA directions of Section 4 as the shared factors,
   remove them from each modality, and run PCA on what is left — the modality's private factors.

In [ ]:
# ── 7.1 Weighted joint PCA (multiple factor analysis) and variance explained per modality ──
def weight_block(A):
    """Centre a block and divide it by its first singular value (multiple factor analysis weighting)."""
    A = A - A.mean(axis=0)
    return A / np.linalg.svd(A, compute_uv=False)[0]


mfa_blocks = {"RNA": weight_block(Z_rna), "ADT": weight_block(adt_clr), "ATAC": weight_block(lsi[:, 1:31])}
J = np.hstack(list(mfa_blocks.values()))
F, loadings, evr_joint = pca(J, n_components=10)
edges = np.cumsum([0] + [b.shape[1] for b in mfa_blocks.values()])
r2 = np.array([[np.sum(np.outer(F[:, k], loadings[k, edges[b]:edges[b + 1]]) ** 2) / np.sum(B ** 2)
                for b, B in enumerate(mfa_blocks.values())] for k in range(10)])
row, labels["joint PCA"] = score_embedding("intermediate: weighted joint PCA (10 factors)", F)
print(row.round(2).to_string())
factor_table = pd.DataFrame(r2, columns=[f"R² {m}" for m in mfa_blocks], index=[f"F{k + 1}" for k in range(10)])
for state, (pop, y) in HIDDEN.items():
    factor_table[f"|r| {state}"] = [abs(np.corrcoef(F[pop, k], y[pop])[0, 1]) for k in range(10)]
print(factor_table.round(2).to_string())

fig, ax = plt.subplots(figsize=(8, 2.8))
im = ax.imshow(r2.T, cmap="Purples", aspect="auto")
ax.set_yticks(range(3)); ax.set_yticklabels(list(mfa_blocks)); ax.set_xticks(range(10))
ax.set_xticklabels([f"F{k + 1}" for k in range(10)]); ax.set_xlabel("joint factor")
plt.colorbar(im, ax=ax, label="fraction of the block's variance")
ax.set_title("SIMULATED: variance explained per factor and modality (a MOFA-style plot)")
plt.tight_layout(); plt.show()

In [ ]:
# ── 7.2 🧪 Shared and private factors: which hidden states live where? ──
shared = cv_rna / cv_rna.std(axis=0) + cv_adt / cv_adt.std(axis=0)        # the 5 shared CCA directions
Q, _ = np.linalg.qr(shared - shared.mean(axis=0))
donor1 = obs.donor.to_numpy(str) == "donor1"
targets = {"memory (T cells)": (is_t, t_state == "memory"), "Treg (CD4 T)": (true_type == "CD4 T", is_treg),
           "primed (monocytes)": (true_type == "Monocyte", is_primed), "donor (all cells)": (np.ones(len(obs), bool), donor1)}
rows = {}
for name, B in mfa_blocks.items():
    resid = B - Q @ (Q.T @ B)                                             # remove the shared space
    P, _, ev = pca(resid, n_components=2)
    for k in range(2):
        rows[f"{name} private factor {k + 1}"] = {"shared fraction of block": 1 - np.sum(resid ** 2) / np.sum(B ** 2),
                                                  **{t: abs(np.corrcoef(P[pop, k], y[pop])[0, 1]) for t, (pop, y) in targets.items()}}
print("🧪 |correlation| of each private factor with the hidden states (inside the relevant population):")
print(pd.DataFrame(rows).T.round(2).to_string())

🔍 **What just happened?** The variance-explained heat map has the typical MOFA look: the first four or five factors
explain variance in all three modalities at once — the shared cell-type structure — and the later ones are weak. But
the factor table shows a limitation of plain PCA: the hidden states do not sit in factors of their own. Memory status
is smeared over F1–F4 and F9 (|r| 0.31–0.71), priming over F1, F3–F7. PCA factors are defined only up to rotation
within a subspace and are chosen for variance, not interpretability; MOFA+'s sparsity priors exist precisely to pull
out factors that are active in one modality only. Clustering in the 10-factor joint space recovers the types and the
primed monocytes but not the 45 Tregs (fine-state ARI 0.87, Treg AUROC 0.49) — a few leading factors summarise the
large sources of variation and can bury a rare state.

The shared/private decomposition is much cleaner. About half of the RNA embedding's variance, 82 % of the protein
block's and only 23 % of the chromatin block's is shared. What remains, privately, is exactly the hidden biology:

- **ADT private factor 1 is memory status** (|r| = 0.98 within T cells) and **private factor 2 is Treg status** (0.89);
- **ATAC private factor 1 is priming** (0.89 within monocytes);
- **RNA private factor 2 is the donor** (0.82): the simulator gives each donor small gene-expression shifts that exist
  only in RNA. A modality-specific factor can be biology *or* a technical/nuisance effect — the model cannot tell which.

That is the interpretive power of joint factor models: a factor that is private to one modality says "this modality
sees something the others do not", and the next step is to find out what. (A caution on our shortcut: we defined
"shared" from RNA and protein only. Shared directions estimated with a modality that carries a state would leak it into
the other modalities' residuals; MOFA+ avoids this by fitting everything in one probabilistic model.)

✏️ **Your turn.** Compute `pca(resid, 3)` for the RNA block and correlate its components with `true_type == "CD8 T"`.
Is anything about cell types left in the RNA residual after removing five shared directions?

---
## 8 · 🧭 From toy to real data

> 📍 **Where we are.** Everything above ran on 2,500 simulated cells with an answer key. Here is what changes with real
> multimodal data — no downloads, a map for your first real analysis.

**Scale and sparsity.** Real multiome data have ~30,000 genes and 100,000–300,000 peaks; a typical nucleus has a few
thousand ATAC fragments, so only a few per cent of the peak matrix is non-zero (our simulation: 4 %). Everything is stored
sparse, LSI is computed with sparse truncated SVD, and metacells or neighbourhood aggregation become essential.
CITE-seq panels range from ten to a few hundred antibodies.

**Public data to practise on.** The 10x Genomics PBMC Multiome and CITE-seq data sets
([10x Genomics datasets](https://www.10xgenomics.com/datasets)); the CITE-seq PBMC reference of Hao *et al.* (2021),
distributed with [Seurat](https://satijalab.org/seurat/) (Azimuth); the NeurIPS 2021 multimodal challenge data
(bone-marrow mononuclear cells from several donors and sites, CITE-seq and Multiome; Lance *et al.* 2022), available
through [GEO](https://www.ncbi.nlm.nih.gov/geo/); multimodal collections in [CELLxGENE](https://cellxgene.cziscience.com/).

**Production tools.**
[Seurat](https://satijalab.org/seurat/) (R) — CLR, WNN (`FindMultiModalNeighbors`), reference mapping;
[Signac](https://stuartlab.org/signac/) (R) — scATAC/multiome: TF-IDF/LSI, gene activity, `LinkPeaks`, motifs;
[ArchR](https://www.archrproject.com/) (R) — large scATAC projects, peak-to-gene links;
[MOFA+](https://biofam.github.io/MOFA2/) (R/Python) — group factor analysis;
[scvi-tools](https://scvi-tools.org/) (Python) — totalVI, MultiVI and related models;
[muon](https://muon.readthedocs.io/) with MuData (Python) — the multimodal counterpart of Scanpy/AnnData;
the `dsb` R package for protein normalisation.

**What goes wrong first.**

1. **Protein background and antibody artefacts.** Ambient antibody, non-specific binding, antibody aggregates and
   saturation differ between antibodies; without empty droplets or isotype controls, background is hard to separate
   from low true expression. Doublets show up as cells positive for mutually exclusive proteins (CD3 and CD19).
2. **Chromatin quality and peaks.** Peak calling (pseudobulk per cluster, then merging), fragments in blacklisted
   regions, TSS enrichment and nucleosome-signal QC, and the choice of peak set all change downstream results; peak sets
   from different studies do not match, which complicates integration across studies.
3. **Unequal information and missing modalities.** Some cells have poor ADT or ATAC; some studies measured only one
   modality (**mosaic integration**). Integration that ignores modality quality can propagate noise; integration that
   only keeps shared variation (CCA-like) can discard what one modality uniquely sees — as Section 4 showed.

**What simulation cannot teach.** Real data have no answer key for "true" states or "true" enhancer–gene links;
validation relies on known immunology (protein gating, as in flow cytometry), consistency across donors, and perturbation
experiments (CRISPR inhibition of enhancers; [Module 41](41_perturbation_causal_inference.ipynb)).

In [ ]:
# ── 8.1 Total run time ──
print(f"⏱ Notebook compute: {time.time() - T_START:.0f} s")

## 📌 Key takeaways

- **Multimodal assays exist because some biology is invisible in RNA:** splice isoforms and post-transcriptional
  regulation show up in protein (naive vs memory T cells, Tregs), regulatory priming in chromatin.
- **Each modality needs its own normalisation:** CLR or a background model (DSB) for antibody tags, whose counts are
  high but sit on a background; log-TF-IDF + LSI for binary chromatin, dropping the depth-driven first component.
- **What a modality sees can be measured** with ARIs and neighbourhood AUROCs against ground truth; here RNA saw types,
  protein saw T-cell states, chromatin saw priming — and none saw everything.
- **Integration choices encode what you call signal:** CCA keeps shared variation and discards modality-specific states;
  unscaled concatenation lets one modality dominate; WNN learns per-cell weights (interpretable, robust to uneven
  quality) but did not beat a well-scaled concatenation on this clean simulation.
- **Cross-modal prediction works only through shared structure:** cell-type proteins are predictable from RNA, CD45RO
  and CD25 are not — which is exactly why they are worth measuring.
- **Peak–gene correlations are confounded by cell type:** co-specific peaks and genes correlate whether or not they are
  linked; metacells reduce noise, distance windows shrink the hypothesis space, and perturbations provide causal tests.
- **Joint factor models** (MOFA+, totalVI, MultiVI) separate shared from modality-specific variation; private factors
  can be biology (memory, priming) or nuisance (donor).

---
## ✅ Checkpoint

1. Why is the CLR transform used for ADT counts, and what artefact can it introduce with a small panel?
   <details><summary>Answer</summary>CLR subtracts the cell's mean log count over all proteins, cancelling a technical factor that multiplies all of a cell's counts (cell size, antibody capture). But the geometric mean depends on how many panel proteins the cell expresses, so negative cells of different types get different CLR values (CD8-negative NK and dendritic cells sat higher than B cells in 2.4).</details>
2. Every cell has about 15 counts of the isotype control. What does that mean for gating a rare protein like CD25?
   <details><summary>Answer</summary>ADT counts sit on a background (ambient and non-specific antibody) present in every cell; a non-zero count does not mean expression. A background model — empty droplets, isotype controls or the negative population, as in DSB — is needed to call positivity.</details>
3. Why do we drop the first LSI component of scATAC data?
   <details><summary>Answer</summary>LSI is an SVD without centring; the first component captures the average accessibility pattern, whose strength per cell tracks sequencing depth (r = 0.95 here). Keeping it makes neighbourhoods depth-matched rather than biology-matched; L2-normalising the remaining coordinates removes most of the rest of the depth signal.</details>
4. What does CCA maximise, and why did CCA-based integration lose the naive/memory distinction?
   <details><summary>Answer</summary>The correlation between a linear score of one modality and a linear score of the other, across cells (the singular values of the whitened cross-covariance). Naive/memory differences exist only in protein, so they are uncorrelated with RNA and CCA discards them; the five shared directions encode cell types.</details>
5. By hand: two modalities give canonical correlations 0.96, 0.94, 0.92, 0.86, 0.79, 0.11, 0.10. How many shared dimensions would you keep, and how many cell types might the data contain?
   <details><summary>Answer</summary>Five (the drop from 0.79 to 0.11 is the boundary to noise). Separating K groups needs K − 1 directions, so about six cell types share structure across the modalities.</details>
6. In WNN, a cell has within-modality affinities θ_RNA = 0.9, θ_ADT = 0.9 and cross-modality affinities 0.85 (RNA predicted from ADT neighbours) and 0.1 (ADT predicted from RNA neighbours). Which modality gets more weight, and why?
   <details><summary>Answer</summary>ADT: its ratio 0.9/0.1 = 9 is much larger than RNA's 0.9/0.85 ≈ 1.06. RNA neighbours fail to predict the cell's protein profile, so protein carries information RNA lacks; ADT neighbours predict the RNA profile almost as well as RNA itself.</details>
7. A protein-from-RNA model reaches an average R² of 0.74 across the panel. Why might it still be useless for your question about T-cell memory?
   <details><summary>Answer</summary>Averages are dominated by cell-type proteins that RNA predicts through the cell type; CD45RO and CD25, which define memory and Tregs, had R² ≈ 0.3 and ≈ 0. Predictions can only reproduce what RNA already implies.</details>
8. Why can a peak correlate strongly with a gene across cells although it does not regulate it? Name two remedies.
   <details><summary>Answer</summary>Cell type (or state) is a common cause: it opens the peaks of all its genes and switches on all its transcripts, so co-specific peaks and genes correlate (the open path peak ← type → gene). Remedies: restrict candidates with a prior such as genomic distance, compare with matched background peaks or within-type variation, and validate by perturbing the enhancer (CRISPRi).</details>

## 🧠 Questions & discussion

1. In our simulation, protein revealed T-cell memory because RNA had no memory signal at all. In real T cells, memory cells do differ in many transcripts. How would you decide whether the protein measurement adds information beyond RNA on a real data set? Design an analysis.
2. WNN-lite did not beat a balanced concatenation here. Under which conditions in real data would you expect per-cell weights to matter most? Can you construct a simulation where WNN clearly wins?
3. Chromatin potential treats chromatin opening as a predictor of future expression. What alternative explanations could make chromatin and RNA disagree in a real cell, and what experiment would distinguish them?
4. CITE-seq panels are designed with prior knowledge of which proteins matter. How does panel design bias what can be discovered, and how would you choose 20 antibodies for a tissue nobody has profiled?
5. Multimodal atlases are increasingly used as references to annotate new data. What could go wrong if a reference was built from donors of a narrow age range or ancestry, and who bears the cost of such errors?

## 🚀 Bonus — going further

- **Impute chromatin from RNA.** Predict each cell's metacell-averaged accessibility of the interferon peaks from its RNA with k-NN regression. Can RNA predict priming? (It should not — explain why.)
- **MOFA-style group factor analysis.** Fit a factor model $X^{(m)} \approx Z W^{(m)\top}$ by alternating least squares with a group-lasso penalty on each factor–modality block of $W$, and check whether shared and private factors emerge without CCA.
- **Isotype-based background subtraction.** Implement the full DSB step 2: a per-cell two-component Gaussian mixture over the cell's proteins to estimate its background mean, combined with the isotype by PCA. Does it reduce the over-correction we saw in 2.3?
- **Two-modality WNN exactly as in Hao et al.** Restrict WNN-lite to RNA + ADT and apply the softmax to the raw ratios. How do the weights and the clustering change?
- **Try the real thing.** Install Seurat (R) or muon (Python) and follow their WNN tutorial on the 10x PBMC multiome or CITE-seq data; compare the learned modality weights per cell type with the pattern in 4.6.

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module: 34 — Multimodal single-cell integration       Date:
Biological question: which cell states are invisible in RNA, and how do RNA, protein and chromatin combine to see them?
Data (SIMULATED): simulate_multiome (2,500 cells; RNA 800 genes, 15 ADTs incl. isotype, ATAC 2,000 binary peaks;
                  truth: true_type, t_state naive/memory, is_treg, is_primed, peaks.linked_gene)
Method: CLR and DSB-lite (ADT); log-TF-IDF + LSI, drop component 1 (ATAC); k-NN + Louvain per modality;
        early/late integration; CCA by SVD; WNN-lite; ridge/k-NN protein prediction; gene activity;
        metacell peak–gene correlation; weighted joint PCA; shared/private factors
Key equation: clr(x)_p = log(1+x_p) − mean_q log(1+x_q);  CCA: SVD of C_XX^(−1/2) C_XY C_YY^(−1/2)
Key parameters: number of PCs/LSI components; k of the k-NN graph; WNN k and ε; ridge α; metacell size; window size
Ground-truth check result: ARI fine per modality …; ARI fine early/CCA/WNN …; R² CD14/CD45RO/CD25 …; link AUROC/AP …
What changed when I modified …: (e.g. dropping the LSI depth component, block scaling, window size)
One failure mode: (e.g. CCA discarding modality-specific states; cell-type confounding of peak–gene links)
Connection to research (paper/tool): Seurat WNN (Hao et al. 2021); Signac; ArchR; MOFA+; totalVI; DSB
```

## 📚 References

- Stoeckius M. *et al.* (2017) Simultaneous epitope and transcriptome measurement in single cells. *Nature Methods* 14:865–868.
- Peterson V. M. *et al.* (2017) Multiplexed quantification of proteins and transcripts in single cells. *Nature Biotechnology* 35:936–939.
- Buenrostro J. D. *et al.* (2013) Transposition of native chromatin for fast and sensitive epigenomic profiling of open chromatin, DNA-binding proteins and nucleosome position. *Nature Methods* 10:1213–1218.
- Buenrostro J. D. *et al.* (2015) Single-cell chromatin accessibility reveals principles of regulatory variation. *Nature* 523:486–490.
- Cusanovich D. A. *et al.* (2015) Multiplex single-cell profiling of chromatin accessibility by combinatorial cellular indexing. *Science* 348:910–914.
- Ma S. *et al.* (2020) Chromatin potential identified by shared single-cell profiling of RNA and chromatin. *Cell* 183:1103–1116.
- Hao Y. *et al.* (2021) Integrated analysis of multimodal single-cell data. *Cell* 184:3573–3587.
- Stuart T. *et al.* (2019) Comprehensive integration of single-cell data. *Cell* 177:1888–1902.
- Hotelling H. (1936) Relations between two sets of variates. *Biometrika* 28:321–377.
- Mulè M. P., Martins A. J., Tsang J. S. (2022) Normalizing and denoising protein expression data from droplet-based single cell profiling. *Nature Communications* 13:2099.
- Gayoso A. *et al.* (2021) Joint probabilistic modeling of single-cell multi-omic data with totalVI. *Nature Methods* 18:272–282.
- Argelaguet R. *et al.* (2018) Multi-Omics Factor Analysis — a framework for unsupervised integration of multi-omics data sets. *Molecular Systems Biology* 14:e8124.
- Argelaguet R. *et al.* (2020) MOFA+: a statistical framework for comprehensive integration of multi-modal single-cell data. *Genome Biology* 21:111.
- Ashuach T. *et al.* (2023) MultiVI: deep generative model for the integration of multimodal data. *Nature Methods* 20:1222–1231.
- Stuart T. *et al.* (2021) Single-cell chromatin state analysis with Signac. *Nature Methods* 18:1333–1341.
- Granja J. M. *et al.* (2021) ArchR is a scalable software package for integrative single-cell chromatin accessibility analysis. *Nature Genetics* 53:403–411.
- Fulco C. P. *et al.* (2019) Activity-by-contact model of enhancer–promoter regulation from thousands of CRISPR perturbations. *Nature Genetics* 51:1664–1669.
- Lance C. *et al.* (2022) Multimodal single cell data integration challenge: results and lessons learned. *Proceedings of the NeurIPS 2021 Competitions and Demonstrations Track*, PMLR 176.
- Efremova M., Teichmann S. A. (2020) Computational methods for single-cell omics across modalities. *Nature Methods* 17:14–17.

## ➡️ Next

Multimodal data showed us cells from several angles; the enhancer–gene links reminded us that correlation is not
regulation. The next part of the course turns to genomes, proteins and molecules, starting with
**[Module 35 · Statistical genetics at scale](35_statistical_genetics.ipynb)** — mixed models, fine-mapping, polygenic
scores and Mendelian randomisation. The causal thread returns in
[Module 41 · Perturbation biology and causal inference](41_perturbation_causal_inference.ipynb).